In [ ]:
%pip install neo4j

In [ ]:
import os
from neo4j import GraphDatabase

URI = os.environ["NEO4J_URI"]
USERNAME = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
)

driver.verify_connectivity()
print("Neo4j connection successful.")

with driver.session(database=DATABASE) as session:
    result = session.run("RETURN 1 AS ok")
    print("Database test:", result.single()["ok"])

In [ ]:

from google.colab import userdata
from neo4j import GraphDatabase

# Read credentials securely from Google Colab Secrets
URI = userdata.get("NEO4J_URI")
USERNAME = userdata.get("NEO4J_USERNAME")
PASSWORD = userdata.get("NEO4J_PASSWORD")

# The database name is optional; defaults to "neo4j"
DATABASE = userdata.get("NEO4J_DATABASE") or "neo4j"

# Check that required secrets exist without displaying their values
missing = [
    name for name, value in [
        ("NEO4J_URI", URI),
        ("NEO4J_USERNAME", USERNAME),
        ("NEO4J_PASSWORD", PASSWORD),
    ]
    if not value or not value.strip()
]

if missing:
    raise ValueError(
        f"Missing or empty Colab Secrets: {', '.join(missing)}. "
        "Check the Secrets panel and enable notebook access."
    )

# Connect to Neo4j AuraDB and test the database
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

try:
    driver.verify_connectivity()
    print("Neo4j connection successful.")

    with driver.session(database=DATABASE) as session:
        result = session.run("RETURN 1 AS ok")
        print("Database test:", result.single()["ok"])

finally:
    driver.close()
    print("Neo4j driver closed.")


In [ ]:
# QueryCraft: Neo4j Database Configuration

# For Neo4j AuraDB, the default database is usually "neo4j".
NEO4J_DATABASE = "neo4j"

print("Neo4j database configured:", NEO4J_DATABASE)

In [ ]:
# Verify that the Neo4j driver exists and is connected

try:
    print("Driver variable exists:", driver is not None)

    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("RETURN 1 AS test")
        print("Connection test:", result.single()["test"])
        print("Neo4j connection successful.")

except NameError as e:
    print("Missing variable:", e)
    print("Run your Neo4j driver initialization cell first.")

except Exception as e:
    print("Neo4j connection or database error:", repr(e))

In [ ]:
# QueryCraft: Recreate Neo4j AuraDB Driver

from neo4j import GraphDatabase
from google.colab import userdata

# Read credentials from Google Colab Secrets.
NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

NEO4J_DATABASE = "57ad8511"

# Create a fresh driver.
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# Verify connectivity.
driver.verify_connectivity()

print("Neo4j driver initialized successfully.")
print("Database configured:", NEO4J_DATABASE)
print("Connectivity verification passed.")

In [ ]:
# QueryCraft: Neo4j Connection Test

with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run("RETURN 1 AS test").single()
    print("Connection test:", record["test"])

print("Neo4j connection successful.")

In [ ]:

# QueryCraft: Live Neo4j Graph Schema Inspection

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== 1. NODE LABELS AND COUNTS ===")
    result = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS node_count
        ORDER BY label
    """)
    for record in result:
        print(record.data())

    print("\n=== 2. RELATIONSHIP TYPES AND COUNTS ===")
    result = session.run("""
        MATCH (a)-[r]->(b)
        RETURN labels(a) AS source_labels,
               type(r) AS relationship,
               labels(b) AS target_labels,
               count(*) AS relationship_count
        ORDER BY relationship
    """)
    for record in result:
        print(record.data())

    print("\n=== 3. NODE PROPERTY NAMES ===")
    result = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS property
        RETURN label, property, count(*) AS occurrences
        ORDER BY label, property
    """)
    for record in result:
        print(record.data())

    print("\n=== 4. EXISTING CONSTRAINTS ===")
    result = session.run("SHOW CONSTRAINTS")
    for record in result:
        print(record.data())

    print("\n=== 5. EXISTING INDEXES ===")
    result = session.run("SHOW INDEXES")
    for record in result:
        print({
            "name": record.get("name"),
            "type": record.get("type"),
            "state": record.get("state"),
            "labelsOrTypes": record.get("labelsOrTypes"),
            "properties": record.get("properties")
        })

print("\nSchema inspection completed.")


In [ ]:

import os
from google.colab import userdata
from neo4j import GraphDatabase

# Load Neo4j credentials from Google Colab Secrets
NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

# Define the database name
NEO4J_DATABASE = userdata.get("NEO4J_DATABASE") or "neo4j"

# Validate required credentials without printing secrets
if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError(
        "Missing Neo4j credentials. Check NEO4J_URI, "
        "NEO4J_USERNAME and NEO4J_PASSWORD in Colab Secrets."
    )

# Create the driver
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# Verify connectivity
driver.verify_connectivity()

print("Neo4j connection successful.")
print("Database selected:", NEO4J_DATABASE)


In [ ]:
print("NEO4J_DATABASE defined:", "NEO4J_DATABASE" in globals())
print("Selected database:", NEO4J_DATABASE)

In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== NODE LABELS ===")
    labels = session.run("""
        CALL db.labels()
        YIELD label
        RETURN label
        ORDER BY label
    """).value()
    print(labels or "No node labels found.")

    print("\n=== RELATIONSHIP TYPES ===")
    rel_types = session.run("""
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType
        ORDER BY relationshipType
    """).value()
    print(rel_types or "No relationship types found.")

    print("\n=== PROPERTY KEYS ===")
    property_keys = session.run("""
        CALL db.propertyKeys()
        YIELD propertyKey
        RETURN propertyKey
        ORDER BY propertyKey
    """).value()
    print(property_keys or "No property keys found.")


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== NODE PROPERTIES ===")
    result = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS property
        RETURN label, property, count(*) AS occurrences
        ORDER BY label, property
    """)
    node_properties = [record.data() for record in result]
    for row in node_properties:
        print(row)
    if not node_properties:
        print("No node properties found, or the graph is empty.")

    print("\n=== RELATIONSHIP PROPERTIES ===")
    result = session.run("""
        MATCH ()-[r]->()
        UNWIND keys(r) AS property
        RETURN type(r) AS relationship_type,
               property,
               count(*) AS occurrences
        ORDER BY relationship_type, property
    """)
    relationship_properties = [record.data() for record in result]
    for row in relationship_properties:
        print(row)
    if not relationship_properties:
        print("No relationship properties found.")


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== NODE COUNTS ===")
    result = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS total
        ORDER BY label
    """)
    rows = [record.data() for record in result]
    for row in rows:
        print(row)
    if not rows:
        print("No labeled nodes found.")

    print("\n=== RELATIONSHIP COUNTS ===")
    result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship_type,
               count(*) AS total
        ORDER BY relationship_type
    """)
    rows = [record.data() for record in result]
    for row in rows:
        print(row)
    if not rows:
        print("No relationships found.")

    print("\n=== TOTAL GRAPH SIZE ===")
    result = session.run("""
        MATCH (n)
        WITH count(n) AS nodes
        MATCH ()-[r]->()
        RETURN nodes, count(r) AS relationships
    """)
    print(result.single().data())


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== CONSTRAINTS ===")
    try:
        result = session.run("SHOW CONSTRAINTS")
        rows = [record.data() for record in result]
        for row in rows:
            print(row)
        if not rows:
            print("No constraints found.")
    except Exception as e:
        print("Could not inspect constraints:", str(e))

    print("\n=== INDEXES ===")
    try:
        result = session.run("SHOW INDEXES")
        rows = [record.data() for record in result]
        for row in rows:
            print(row)
        if not rows:
            print("No indexes found.")
    except Exception as e:
        print("Could not inspect indexes:", str(e))


In [ ]:

EXPECTED_LABELS = {"Movie", "Person", "Genre"}
EXPECTED_RELATIONSHIPS = {"ACTED_IN", "DIRECTED", "IN_GENRE"}

with driver.session(database=NEO4J_DATABASE) as session:
    actual_labels = set(session.run(
        "CALL db.labels() YIELD label RETURN label"
    ).value())

    actual_relationships = set(session.run(
        """CALL db.relationshipTypes()
           YIELD relationshipType
           RETURN relationshipType"""
    ).value())

    print("=== SCHEMA VALIDATION ===")

    print("Expected labels:", sorted(EXPECTED_LABELS))
    print("Actual labels:", sorted(actual_labels))
    print("Missing labels:", sorted(EXPECTED_LABELS - actual_labels))
    print("Unexpected labels:", sorted(actual_labels - EXPECTED_LABELS))

    print("\nExpected relationship types:",
          sorted(EXPECTED_RELATIONSHIPS))
    print("Actual relationship types:",
          sorted(actual_relationships))
    print("Missing relationships:",
          sorted(EXPECTED_RELATIONSHIPS - actual_relationships))
    print("Unexpected relationships:",
          sorted(actual_relationships - EXPECTED_RELATIONSHIPS))

    schema_ok = (
        EXPECTED_LABELS == actual_labels
        and EXPECTED_RELATIONSHIPS == actual_relationships
    )
    print("\nSchema names match expected:", schema_ok)


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== ACTUAL NODES ===")
    result = session.run("""
        MATCH (n)
        RETURN labels(n) AS labels,
               properties(n) AS properties
        ORDER BY labels
    """)
    for record in result:
        print(record.data())

    print("\n=== ACTUAL RELATIONSHIPS ===")
    result = session.run("""
        MATCH (a)-[r]->(b)
        RETURN labels(a) AS source_labels,
               properties(a) AS source_properties,
               type(r) AS relationship,
               properties(r) AS relationship_properties,
               labels(b) AS target_labels,
               properties(b) AS target_properties
        ORDER BY relationship
    """)
    for record in result:
        print(record.data())


In [ ]:

test_queries = [
    {
        "name": "Find a movie by title",
        "cypher": """
            MATCH (m:Movie {title: 'The Matrix'})
            RETURN m.title AS title, m.released AS released
        """
    },
    {
        "name": "Find actors in a movie",
        "cypher": """
            MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
            RETURN p.name AS person, m.title AS movie
        """
    },
    {
        "name": "Find a movie's genre",
        "cypher": """
            MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
            RETURN m.title AS movie, g.name AS genre
        """
    },
    {
        "name": "Find movies released after 1995",
        "cypher": """
            MATCH (m:Movie)
            WHERE m.released > 1995
            RETURN m.title AS title, m.released AS released
            ORDER BY m.released DESC
        """
    }
]

with driver.session(database=NEO4J_DATABASE) as session:
    for test in test_queries:
        print(f"\n=== {test['name']} ===")
        try:
            records = session.run(test["cypher"]).data()
            print(records if records else "Query executed; no matching records.")
        except Exception as e:
            print("FAILED:", str(e))

print("\nBasic Cypher tests completed.")


In [ ]:
!python C:\Users\Pranay\Downloads\querycraft_synthetic_dataset\querycraft_dataset\validate_dataset.py

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
from pathlib import Path

zip_path = "/content/querycraft_synthetic_dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

dataset_dir = Path("/content/querycraft_dataset")

print("Dataset directory exists:", dataset_dir.exists())
print("Files found:")

for file in sorted(dataset_dir.iterdir()):
    print(file.name)

In [ ]:
!python /content/querycraft_dataset/validate_dataset.py

In [ ]:
!find /content -name validate_dataset.py

In [ ]:

import json
from pathlib import Path

dataset_dir = Path("/content/querycraft_dataset")

def load_jsonl(filename):
    with open(dataset_dir / filename, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl("train.jsonl")
eval_data = load_jsonl("eval.jsonl")

def normalize(rows):
    # Compare result rows without relying on row order.
    # Dictionary key/value types are retained.
    return sorted(
        (json.dumps(row, sort_keys=True, default=str) for row in rows)
    )

def validate_split(session, examples, split_name):
    passed = []
    failed = []

    for example in examples:
        try:
            actual = session.run(example["cypher"]).data()
            expected = example["expected_result"]

            if normalize(actual) == normalize(expected):
                passed.append(example["id"])
            else:
                failed.append({
                    "id": example["id"],
                    "question": example["question"],
                    "expected": expected,
                    "actual": actual,
                    "error": "Result mismatch"
                })

        except Exception as e:
            failed.append({
                "id": example["id"],
                "question": example["question"],
                "expected": example["expected_result"],
                "actual": None,
                "error": str(e)
            })

    print(f"\n=== {split_name} ===")
    print("Total:", len(examples))
    print("Passed:", len(passed))
    print("Failed:", len(failed))

    for item in failed:
        print("\nFAILED:", item["id"], "-", item["question"])
        print("Expected:", item["expected"])
        print("Actual:", item["actual"])
        print("Error:", item["error"])

    return passed, failed

# Use read-only access for this validation.
with driver.session(
    database=NEO4J_DATABASE,
    default_access_mode="READ"
) as session:
    train_passed, train_failed = validate_split(
        session, train_data, "TRAINING CANDIDATES"
    )

    eval_passed, eval_failed = validate_split(
        session, eval_data, "HELD-OUT EVALUATION CANDIDATES"
    )

print("\n=== VALIDATION SUMMARY ===")
print("Training query checks passed:", len(train_passed), "/", len(train_data))
print("Evaluation query checks passed:", len(eval_passed), "/", len(eval_data))
print("Total failures:", len(train_failed) + len(eval_failed))
print("All expected results matched:",
      not train_failed and not eval_failed)


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

import json
import shutil
from pathlib import Path

# Existing dataset folder in Google Colab
SOURCE_DIR = Path("/content/querycraft_dataset")

# Destination folder in Google Drive
DEST_DIR = Path("/content/drive/MyDrive/content/querycraft_dataset")
DEST_DIR.mkdir(parents=True, exist_ok=True)

source_train = SOURCE_DIR / "train.jsonl"
source_eval = SOURCE_DIR / "eval.jsonl"

destination_train = DEST_DIR / "train.jsonl"
destination_eval = DEST_DIR / "evaluation.jsonl"

# Check that source files exist before copying
for path in [source_train, source_eval]:
    if not path.exists():
        raise FileNotFoundError(
            f"Cannot find {path}. Check your source dataset folder."
        )

# Copy existing reviewed datasets
shutil.copy2(source_train, destination_train)
shutil.copy2(source_eval, destination_eval)

print("Files saved successfully!")
print("Training file:", destination_train)
print("Evaluation file:", destination_eval)
print("Training file size:", destination_train.stat().st_size, "bytes")
print("Evaluation file size:", destination_eval.stat().st_size, "bytes")


In [ ]:

import json
from pathlib import Path

DEST_DIR = Path("/content/drive/MyDrive/content/querycraft_dataset")

train_path = DEST_DIR / "train.jsonl"
eval_path = DEST_DIR / "evaluation.jsonl"

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_jsonl(train_path)
evaluation = load_jsonl(eval_path)

print("Training examples:", len(train))
print("Evaluation examples:", len(evaluation))
print("Training file exists:", train_path.exists())
print("Evaluation file exists:", eval_path.exists())

assert len(train) == 30, f"Expected 30, got {len(train)}"
assert len(evaluation) == 10, f"Expected 10, got {len(evaluation)}"

train_ids = {row["id"] for row in train}
eval_ids = {row["id"] for row in evaluation}

assert len(train_ids) == 30, "Duplicate training IDs"
assert len(eval_ids) == 10, "Duplicate evaluation IDs"
assert not train_ids.intersection(eval_ids), "Train/evaluation ID leakage"

print("File and split validation: PASS")


In [ ]:
from pathlib import Path

for p in Path("/content").rglob("*.jsonl"):
    print(p)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DEST_DIR = Path("/content/drive/MyDrive/content/querycraft_dataset")
DEST_DIR.mkdir(parents=True, exist_ok=True)

train_path = DEST_DIR / "train.jsonl"
eval_path = DEST_DIR / "evaluation.jsonl"

print("Training file:", train_path)
print("Evaluation file:", eval_path)

In [ ]:

import json
import csv
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")
ROOT.mkdir(parents=True, exist_ok=True)

# Existing source files
train_jsonl = ROOT / "train.jsonl"
eval_jsonl = ROOT / "evaluation.jsonl"

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def convert_jsonl_to_csv(source, destination):
    rows = read_jsonl(source)

    if not rows:
        raise ValueError(f"No records found in {source}")

    columns = [
        "id", "question", "cypher",
        "expected_result", "category", "review_status"
    ]

    with open(destination, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()

        for row in rows:
            record = {key: row.get(key) for key in columns}

            # Preserve expected_result as valid JSON in one CSV cell
            record["expected_result"] = json.dumps(
                row["expected_result"], ensure_ascii=False
            )
            writer.writerow(record)

    print(f"Created: {destination} ({len(rows)} records)")

convert_jsonl_to_csv(train_jsonl, ROOT / "train.csv")
convert_jsonl_to_csv(eval_jsonl, ROOT / "eval.csv")


In [ ]:
ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

In [ ]:

import csv
from pathlib import Path
from collections import Counter

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

for filename in ["train.csv", "eval.csv"]:
    path = ROOT / filename

    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    print(f"\n--- {filename} ---")
    print("Review status counts:")

    counts = Counter(
        (row.get("review_status") or "").strip()
        for row in rows
    )
    print(counts)

    for row in rows:
        status = (row.get("review_status") or "").strip()
        if status != "APPROVED":
            print(
                "ID:", row.get("id"),
                "| Status:", repr(status),
                "| Question:", row.get("question")
            )


In [ ]:

import csv
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

files = [
    ROOT / "train.csv",
    ROOT / "eval.csv",
]

for path in files:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

# Validate both files before modifying either one
for path in files:
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        if not reader.fieldnames or "review_status" not in reader.fieldnames:
            raise ValueError(
                f"{path.name} does not contain a review_status column."
            )

# Update the review_status column and save the files
for path in files:
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    for row in rows:
        row["review_status"] = "APPROVED"

    # Write a temporary file first, then replace the original
    temp_path = path.with_name(path.name + ".tmp")

    with open(temp_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    temp_path.replace(path)

    print(f"Updated: {path.name}")
    print(f"Records marked APPROVED: {len(rows)}")

print("\nBoth CSV files have been updated in Google Drive.")


In [ ]:

import csv
from collections import Counter
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

for filename in ["train.csv", "eval.csv"]:
    path = ROOT / filename

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    statuses = Counter(
        row["review_status"] for row in rows
    )

    print(f"\n{filename}")
    print("Total records:", len(rows))
    print("Review statuses:", dict(statuses))

    assert all(
        row["review_status"] == "APPROVED" for row in rows
    ), f"Some records in {filename} are not approved."

print("\nStatus verification: PASS")


In [ ]:

from google.colab import userdata
from neo4j import GraphDatabase

NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")
NEO4J_DATABASE = userdata.get("NEO4J_DATABASE") or "neo4j"

if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError(
        "Missing Neo4j credentials in Google Colab Secrets."
    )

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

driver.verify_connectivity()
print("Neo4j connectivity: PASS")

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(
        "RETURN 1 AS connection_test"
    ).single()
    print("Database test:", result["connection_test"])


In [ ]:

import csv
import json
import re
from pathlib import Path
from collections.abc import Mapping

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

TRAIN_FILE = ROOT / "train.csv"
EVAL_FILE = ROOT / "eval.csv"
REPORT_FILE = ROOT / "validation_report.csv"

def load_dataset(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    for row in rows:
        row["expected_result"] = json.loads(
            row["expected_result"]
        )

    return rows

train = load_dataset(TRAIN_FILE)
eval_set = load_dataset(EVAL_FILE)

# Conservative allowlist for read-only QueryCraft queries.
WRITE_OR_UNSAFE = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|"
    r"FOREACH|CALL|LOAD\s+CSV|GRANT|DENY|REVOKE|"
    r"TERMINATE|ALTER|RENAME|START\s+DATABASE|"
    r"STOP\s+DATABASE|CREATE\s+DATABASE|DROP\s+DATABASE|"
    r"USE|SHOW)\b",
    re.IGNORECASE,
)

def check_query_safety(query):
    query = query.strip()

    if not query:
        raise ValueError("Empty Cypher query")

    # Permit only queries beginning with MATCH or OPTIONAL MATCH.
    if not re.match(
        r"^(MATCH|OPTIONAL\s+MATCH)\b",
        query,
        re.IGNORECASE,
    ):
        raise ValueError(
            "Query must start with MATCH or OPTIONAL MATCH"
        )

    if WRITE_OR_UNSAFE.search(query):
        raise ValueError(
            "Query contains a prohibited write or unsafe construct"
        )

    if ";" in query:
        raise ValueError(
            "Semicolons/multiple statements are not permitted"
        )

def normalize(value):
    # Convert Neo4j nodes and other mappings into ordinary dictionaries.
    if isinstance(value, Mapping):
        return {
            str(k): normalize(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [normalize(v) for v in value]

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    # Support Neo4j temporal/spatial values if returned.
    return str(value)

def canonical(value):
    return json.dumps(
        normalize(value),
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )

def validate_one(row, split_name):
    record_id = row.get("id", "")
    query = row.get("cypher", "")

    try:
        if row.get("review_status", "").strip() != "APPROVED":
            raise ValueError("Record is not marked APPROVED")

        check_query_safety(query)

        expected = row["expected_result"]

        if not isinstance(expected, list):
            raise ValueError("expected_result must be a JSON list")

        # execute_read uses a read transaction.
        # Neo4j rejects ordinary database writes in this transaction.
        def run_read_query(tx):
            result = tx.run(query)
            return [record.data() for record in result]

        with driver.session(database=NEO4J_DATABASE) as session:
            actual = session.execute_read(run_read_query)

        # Compare as unordered rows, so row ordering alone does not
        # cause a mismatch. Values and duplicate rows are still checked.
        expected_sorted = sorted(canonical(x) for x in expected)
        actual_sorted = sorted(canonical(x) for x in actual)

        passed = expected_sorted == actual_sorted

        return {
            "split": split_name,
            "id": record_id,
            "status": "PASS" if passed else "RESULT_MISMATCH",
            "expected_count": len(expected),
            "actual_count": len(actual),
            "error": "",
            "expected_result": json.dumps(
                expected, ensure_ascii=False, default=str
            ),
            "actual_result": json.dumps(
                normalize(actual), ensure_ascii=False, default=str
            ),
        }

    except Exception as exc:
        return {
            "split": split_name,
            "id": record_id,
            "status": "ERROR_OR_REJECTED",
            "expected_count": (
                len(row["expected_result"])
                if isinstance(row.get("expected_result"), list)
                else ""
            ),
            "actual_count": "",
            "error": f"{type(exc).__name__}: {exc}",
            "expected_result": json.dumps(
                row.get("expected_result"), ensure_ascii=False,
                default=str
            ),
            "actual_result": "",
        }

report = []

for split_name, rows in [
    ("train", train),
    ("eval", eval_set),
]:
    for row in rows:
        result = validate_one(row, split_name)
        report.append(result)

        print(
            f"{result['split']:5} | "
            f"{result['id']:8} | "
            f"{result['status']}"
        )

# Save the detailed report to Google Drive.
fields = [
    "split", "id", "status",
    "expected_count", "actual_count", "error",
    "expected_result", "actual_result",
]

with open(REPORT_FILE, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(report)

counts = {}
for item in report:
    counts[item["status"]] = counts.get(item["status"], 0) + 1

print("\n=== Validation Summary ===")
print("Total records:", len(report))
print("Results:", counts)
print("Report saved to:", REPORT_FILE)

driver.close()


In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        )
    )
else:
    print("Enable a GPU: Runtime > Change runtime type > T4 GPU")


In [ ]:

%pip install -q -U transformers peft accelerate datasets


In [ ]:
%pip uninstall -y pyarrow

In [ ]:

from google.colab import drive
from huggingface_hub import notebook_login

drive.mount("/content/drive")

# Enter your Hugging Face token in the secure login prompt.
notebook_login()


In [ ]:

import csv
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

def read_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    for row in rows:
        row["expected_result"] = json.loads(
            row["expected_result"]
        )

    return rows

train_rows = read_csv(ROOT / "train.csv")
eval_rows = read_csv(ROOT / "eval.csv")

assert len(train_rows) == 30, len(train_rows)
assert len(eval_rows) == 10, len(eval_rows)

print("Training examples:", len(train_rows))
print("Evaluation examples:", len(eval_rows))
print("Training example:", train_rows[0]["id"])
print("Evaluation example:", eval_rows[0]["id"])


In [ ]:

MODEL_ID = "google/gemma-3-1b-it"

GRAPH_SCHEMA = """
Node labels:
- Movie
- Person
- Genre

Relationship types:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)

Use only the labels, relationships, and properties
that actually exist in the database schema.
"""

def make_prompt(question):
    return f"""You translate natural-language questions into Cypher.

Graph schema:
{GRAPH_SCHEMA}

Rules:
- Generate a read-only Cypher query.
- Use only the supplied schema.
- Do not create, update, or delete data.
- Do not invent node properties.
- Return only the Cypher query, without Markdown fences or explanation.

Question: {question}

Cypher:
"""


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "Enable a GPU before loading the model."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

def generate_cypher(model, question, max_new_tokens=160):
    prompt = make_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

# Test one evaluation example
sample = eval_rows[0]

print("Question:", sample["question"])
print("\nExpected Cypher:\n", sample["cypher"])
print("\nBaseline generated Cypher:\n")
print(generate_cypher(model, sample["question"]))


In [ ]:

baseline_rows = []

for row in eval_rows:
    prediction = generate_cypher(model, row["question"])

    baseline_rows.append({
        "id": row["id"],
        "question": row["question"],
        "expected_cypher": row["cypher"],
        "predicted_cypher": prediction,
        "expected_result": json.dumps(
            row["expected_result"], ensure_ascii=False
        ),
    })

baseline_path = ROOT / "baseline_predictions.csv"

with open(
    baseline_path, "w", encoding="utf-8-sig", newline=""
) as f:
    writer = csv.DictWriter(
        f, fieldnames=baseline_rows[0].keys()
    )
    writer.writeheader()
    writer.writerows(baseline_rows)

print("Saved baseline predictions:", baseline_path)
print("Evaluation questions processed:", len(baseline_rows))


In [ ]:

from datasets import Dataset

def format_training_example(row):
    prompt = make_prompt(row["question"])
    answer = row["cypher"].strip()

    # EOS tells the model where the answer ends.
    return {
        "text": prompt + answer + tokenizer.eos_token
    }

train_dataset = Dataset.from_list([
    format_training_example(row) for row in train_rows
])

print("Training records:", len(train_dataset))
print(train_dataset[0]["text"])


In [ ]:

MAX_LENGTH = 512

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

print("Tokenized examples:", len(tokenized_train))
print("First example tokens:", len(tokenized_train[0]["input_ids"]))

assert all(
    len(x["input_ids"]) > 0 for x in tokenized_train
)


In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

# Ensure the model is in training mode.
model.train()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:

%pip uninstall -y torchao


In [ ]:

import torch
import peft
import transformers
import importlib.util

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.train()
print("Base model reloaded successfully.")


In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:

from pathlib import Path
import csv
import json

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

def read_dataset(filename):
    with open(
        ROOT / filename,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:
        rows = list(csv.DictReader(f))

    for row in rows:
        row["expected_result"] = json.loads(
            row["expected_result"]
        )

    return rows

train_rows = read_dataset("train.csv")
eval_rows = read_dataset("eval.csv")

assert len(train_rows) == 30, len(train_rows)
assert len(eval_rows) == 10, len(eval_rows)

assert all(
    row["review_status"] == "APPROVED"
    for row in train_rows + eval_rows
)

train_ids = {row["id"] for row in train_rows}
eval_ids = {row["id"] for row in eval_rows}

assert train_ids.isdisjoint(eval_ids), "Dataset leakage detected"

print("Training data:", len(train_rows))
print("Evaluation data:", len(eval_rows))
print("Dataset validation: PASS")


In [ ]:

from datasets import Dataset

MAX_LENGTH = 512

def format_training_example(row):
    prompt = make_prompt(row["question"])
    answer = row["cypher"].strip()

    return {
        "text": prompt + answer + tokenizer.eos_token
    }

train_dataset = Dataset.from_list([
    format_training_example(row)
    for row in train_rows
])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

print("Tokenized training records:", len(tokenized_train))
print("First record token count:", len(tokenized_train[0]["input_ids"]))

assert len(tokenized_train) == 30


In [ ]:
%pip install pyarrow

In [ ]:

GRAPH_SCHEMA = """
Node labels:
- Movie
- Person
- Genre

Relationship types:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)

Use only properties that exist in the actual database schema.
"""

def make_prompt(question):
    return f"""You translate natural-language questions into Cypher.

Graph schema:
{GRAPH_SCHEMA}

Rules:
- Generate a read-only Cypher query.
- Use only the supplied schema.
- Do not create, update, or delete data.
- Do not invent node properties.
- Return only the Cypher query, without Markdown fences or explanation.

Question: {question}

Cypher:
"""

print(make_prompt("Which movies were directed by a person named Christopher Nolan?"))


In [ ]:

from datasets import Dataset

MAX_LENGTH = 512

def format_training_example(row):
    prompt = make_prompt(row["question"])
    answer = row["cypher"].strip()

    return {
        "text": prompt + answer + tokenizer.eos_token
    }

train_dataset = Dataset.from_list([
    format_training_example(row)
    for row in train_rows
])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

print("Tokenized training records:", len(tokenized_train))
print(
    "First record token count:",
    len(tokenized_train[0]["input_ids"])
)

assert len(tokenized_train) == 30
print("Training dataset preparation: PASS")


In [ ]:

from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

OUTPUT_DIR = str(ROOT / "gemma3_1b_querycraft_lora")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_torch",
    seed=42,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

train_result = trainer.train()

print("Fine-tuning completed.")
print(train_result.metrics)


In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")

ADAPTER_DIR = ROOT / "querycraft_gemma3_lora_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter weights and configuration
trainer.model.save_pretrained(
    str(ADAPTER_DIR),
    safe_serialization=True,
)

# Save tokenizer files for convenient reuse
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("LoRA adapter saved successfully!")
print("Location:", ADAPTER_DIR)

print("\nSaved files:")
for path in sorted(ADAPTER_DIR.iterdir()):
    print(path.name, "-", path.stat().st_size, "bytes")


In [ ]:

from pathlib import Path

ADAPTER_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_dataset/"
    "querycraft_gemma3_lora_adapter"
)

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

for filename in required_files:
    file_path = ADAPTER_DIR / filename
    print(filename, ":", file_path.exists())

assert all(
    (ADAPTER_DIR / filename).exists()
    for filename in required_files
), "Adapter files are missing. Check whether training and saving completed."

print("Adapter file verification: PASS")


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "google/gemma-3-1b-it"

ADAPTER_DIR = (
    "/content/drive/MyDrive/content/querycraft_dataset/"
    "querycraft_gemma3_lora_adapter"
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

model.eval()

print("Base Gemma model loaded.")
print("LoRA adapter loaded.")


In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact folder:", ARTIFACT_DIR)
print("Folder exists:", ARTIFACT_DIR.exists())


In [ ]:

import shutil
from pathlib import Path

SOURCE_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_dataset"
)
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Files to preserve
files_to_copy = [
    "train.csv",
    "eval.csv",
    "baseline_predictions.csv",
    "finetuned_predictions.csv",
    "validation_report.csv",
]

# Copy available files
copied = []
missing = []

for filename in files_to_copy:
    source = SOURCE_DIR / filename
    destination = ARTIFACT_DIR / filename

    if source.is_file():
        shutil.copy2(source, destination)
        copied.append(filename)
        print("COPIED:", filename)
    else:
        missing.append(filename)
        print("NOT FOUND:", source)

# Copy the saved LoRA adapter directory
adapter_source = SOURCE_DIR / "querycraft_gemma3_lora_adapter"
adapter_destination = ARTIFACT_DIR / "querycraft_gemma3_lora_adapter"

if adapter_source.is_dir():
    if adapter_destination.exists():
        shutil.rmtree(adapter_destination)

    shutil.copytree(adapter_source, adapter_destination)
    print("COPIED ADAPTER:", adapter_destination)
else:
    print("ADAPTER NOT FOUND:", adapter_source)

print("\nFiles copied:", len(copied))
print("Missing CSV files:", missing)


In [ ]:

from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)

expected_files = [
    "train.csv",
    "eval.csv",
    "baseline_predictions.csv",
    "finetuned_predictions.csv",
    "validation_report.csv",
]

for filename in expected_files:
    path = ARTIFACT_DIR / filename
    status = (
        f"OK ({path.stat().st_size:,} bytes)"
        if path.is_file()
        else "MISSING"
    )
    print(f"{filename}: {status}")

adapter_dir = ARTIFACT_DIR / "querycraft_gemma3_lora_adapter"

print("\nLoRA adapter directory:", adapter_dir.exists())

if adapter_dir.is_dir():
    for filename in [
        "adapter_config.json",
        "adapter_model.safetensors",
    ]:
        print(
            f"{filename}:",
            (adapter_dir / filename).is_file()
        )

print("\nAll artifacts currently in the destination:")
for path in sorted(ARTIFACT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(ARTIFACT_DIR))


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from pathlib import Path

BASE_MODEL = "google/gemma-3-1b-it"

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")
ADAPTER_DIR = ROOT / "querycraft_gemma3_lora_adapter"

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"LoRA adapter not found: {ADAPTER_DIR}\n"
        "Save your trained adapter before proceeding."
    )

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
)

model.eval()

print("Fine-tuned QueryCraft model loaded successfully.")


In [ ]:

import csv
import json
import torch
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")
EVAL_FILE = ROOT / "eval.csv"

# Save directly to the final artifact folder
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "finetuned_predictions.csv"

if not EVAL_FILE.exists():
    raise FileNotFoundError(f"Evaluation file not found: {EVAL_FILE}")

with open(EVAL_FILE, encoding="utf-8-sig", newline="") as f:
    eval_rows = list(csv.DictReader(f))

if not eval_rows:
    raise ValueError("eval.csv contains no evaluation records.")

def generate_cypher(model, question, max_new_tokens=160):
    prompt = make_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0, prompt_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

prediction_rows = []

for index, row in enumerate(eval_rows, start=1):
    question = row["question"].strip()

    try:
        prediction = generate_cypher(model, question)

        prediction_rows.append({
            "id": row["id"],
            "question": question,
            "expected_cypher": row["cypher"],
            "predicted_cypher": prediction,
            "expected_result": row["expected_result"],
            "prediction_status": (
                "GENERATED" if prediction else "EMPTY_OUTPUT"
            ),
        })

        print(f"{index}/{len(eval_rows)} completed: {row['id']}")

    except Exception as exc:
        prediction_rows.append({
            "id": row.get("id", ""),
            "question": question,
            "expected_cypher": row.get("cypher", ""),
            "predicted_cypher": "",
            "expected_result": row.get("expected_result", ""),
            "prediction_status": f"ERROR: {type(exc).__name__}: {exc}",
        })

        print(f"{index}/{len(eval_rows)} failed: {row.get('id')}")

fieldnames = [
    "id",
    "question",
    "expected_cypher",
    "predicted_cypher",
    "expected_result",
    "prediction_status",
]

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8-sig",
    newline="",
) as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(prediction_rows)

print("\nPrediction generation finished.")
print("Rows saved:", len(prediction_rows))
print("Output file:", OUTPUT_FILE)
print(
    "Successful generations:",
    sum(r["prediction_status"] == "GENERATED" for r in prediction_rows)
)


In [ ]:

import csv
from pathlib import Path

OUTPUT_FILE = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts/"
    "finetuned_predictions.csv"
)

assert OUTPUT_FILE.is_file(), "Prediction CSV was not created."

with open(OUTPUT_FILE, encoding="utf-8-sig", newline="") as f:
    saved_rows = list(csv.DictReader(f))

print("File exists:", OUTPUT_FILE.exists())
print("File size:", OUTPUT_FILE.stat().st_size, "bytes")
print("Saved prediction rows:", len(saved_rows))
print("Columns:", list(saved_rows[0].keys()) if saved_rows else [])

for row in saved_rows:
    print(
        row["id"],
        "|",
        row["prediction_status"],
        "|",
        row["predicted_cypher"][:120],
    )

assert len(saved_rows) == len(eval_rows), (
    "Saved row count does not match evaluation dataset."
)

print("\nCSV verification: PASS")


In [ ]:

from google.colab import userdata
from neo4j import GraphDatabase
from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)

PREDICTIONS_FILE = ARTIFACT_DIR / "finetuned_predictions.csv"
REPORT_FILE = ARTIFACT_DIR / "validation_report.csv"

NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")
NEO4J_DATABASE = userdata.get("NEO4J_DATABASE") or "neo4j"

if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError(
        "Missing Neo4j credentials. Check your Colab Secrets."
    )

if not PREDICTIONS_FILE.is_file():
    raise FileNotFoundError(
        f"Predictions file not found: {PREDICTIONS_FILE}"
    )

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

driver.verify_connectivity()

with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run(
        "RETURN 1 AS connection_test"
    ).single()
    assert record["connection_test"] == 1

print("Neo4j connection: PASS")
print("Development database connectivity: PASS")
print("Predictions file found:", PREDICTIONS_FILE.name)


In [ ]:

import csv
import json
import re
from collections import Counter
from collections.abc import Mapping

# Conservative policy: allow queries starting with MATCH or OPTIONAL MATCH.
# Reject procedures, writes, multiple statements, and comment syntax.
UNSAFE_KEYWORDS = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|"
    r"CALL|LOAD|FOREACH|GRANT|DENY|REVOKE|TERMINATE|"
    r"ALTER|RENAME|SHOW|USE|START|STOP|APOC)\b",
    re.IGNORECASE,
)

def check_read_query(query):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Empty Cypher query")

    query = query.strip()

    if not re.match(
        r"^(MATCH|OPTIONAL\s+MATCH)\b",
        query,
        re.IGNORECASE,
    ):
        raise ValueError(
            "Rejected: query must start with MATCH or OPTIONAL MATCH"
        )

    if ";" in query:
        raise ValueError("Rejected: semicolons are not permitted")

    if "//" in query or "/*" in query or "*/" in query:
        raise ValueError("Rejected: comments are not permitted")

    if UNSAFE_KEYWORDS.search(query):
        raise ValueError(
            "Rejected: potentially unsafe keyword or procedure detected"
        )

def normalize(value):
    # Neo4j Node objects expose labels and properties.
    if hasattr(value, "labels") and hasattr(value, "items"):
        return {
            "labels": sorted(list(value.labels)),
            "properties": {
                str(k): normalize(v)
                for k, v in value.items()
            },
        }

    # Neo4j relationships expose type and properties.
    if hasattr(value, "type") and hasattr(value, "items"):
        return {
            "type": value.type,
            "properties": {
                str(k): normalize(v)
                for k, v in value.items()
            },
        }

    if isinstance(value, Mapping):
        return {
            str(k): normalize(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [normalize(v) for v in value]

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    # Handles Neo4j temporal/spatial values if present.
    return str(value)

def canonical(value):
    return json.dumps(
        normalize(value),
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )

def parse_expected(raw):
    if isinstance(raw, str):
        return json.loads(raw)
    return raw

def validate_prediction(row):
    record_id = row.get("id", "")
    query = row.get("predicted_cypher", "")

    result = {
        "id": record_id,
        "question": row.get("question", ""),
        "predicted_cypher": query,
        "expected_result": row.get("expected_result", ""),
        "actual_result": "",
        "expected_count": "",
        "actual_count": "",
        "status": "",
        "error": "",
    }

    try:
        expected = parse_expected(row.get("expected_result"))

        if not isinstance(expected, list):
            raise ValueError(
                "expected_result must decode to a JSON list"
            )

        result["expected_count"] = len(expected)

        # Never execute a query that fails the safety checks.
        check_read_query(query)

        # Execute inside a read transaction.
        def execute_read(tx):
            records = tx.run(query)
            return [record.data() for record in records]

        with driver.session(database=NEO4J_DATABASE) as session:
            actual = session.execute_read(execute_read)

        result["actual_count"] = len(actual)
        result["actual_result"] = json.dumps(
            normalize(actual),
            ensure_ascii=False,
            default=str,
        )

        # Compare row contents without depending on row ordering.
        # Counter preserves duplicate rows.
        expected_rows = Counter(canonical(x) for x in expected)
        actual_rows = Counter(canonical(x) for x in actual)

        if expected_rows == actual_rows:
            result["status"] = "PASS"
        else:
            result["status"] = "RESULT_MISMATCH"

    except Exception as exc:
        result["status"] = "ERROR_OR_REJECTED"
        result["error"] = f"{type(exc).__name__}: {exc}"

    return result


with open(
    PREDICTIONS_FILE,
    "r",
    encoding="utf-8-sig",
    newline="",
) as f:
    predictions = list(csv.DictReader(f))

if not predictions:
    raise ValueError("No prediction rows found.")

required_columns = {
    "id", "question", "predicted_cypher", "expected_result"
}
missing_columns = required_columns - set(predictions[0].keys())

if missing_columns:
    raise ValueError(
        f"Prediction CSV is missing columns: {missing_columns}"
    )

report = []

for i, row in enumerate(predictions, start=1):
    outcome = validate_prediction(row)
    report.append(outcome)

    print(
        f"{i}/{len(predictions)} | "
        f"{outcome['id']} | {outcome['status']}"
    )

fields = [
    "id",
    "question",
    "predicted_cypher",
    "expected_result",
    "actual_result",
    "expected_count",
    "actual_count",
    "status",
    "error",
]

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8-sig",
    newline="",
) as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(report)

print("\n=== Neo4j Validation Summary ===")
for status, count in sorted(
    Counter(r["status"] for r in report).items()
):
    print(f"{status}: {count}")

print("Total predictions:", len(report))
print("Validation report saved:", REPORT_FILE)


In [ ]:

failed = [
    row for row in report
    if row["status"] != "PASS"
]

print("Failed or rejected queries:", len(failed))

for row in failed:
    print("\nID:", row["id"])
    print("Status:", row["status"])
    print("Question:", row["question"])
    print("Generated Cypher:", row["predicted_cypher"])
    print("Expected result:", row["expected_result"])
    print("Actual result:", row["actual_result"])
    print("Error:", row["error"])


In [ ]:

import torch

def generate_cypher_clean(model, question, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": (
                "Generate exactly one read-only Cypher query for the "
                "question below.\n\n"
                f"Graph schema:\n{GRAPH_SCHEMA}\n\n"
                "Rules:\n"
                "- Output only one Cypher query.\n"
                "- Do not output Markdown fences.\n"
                "- Do not repeat the question.\n"
                "- Do not include explanations or Python code.\n"
                "- Use only the supplied node labels and relationships.\n"
                "- Do not invent properties.\n"
                "- Use valid Neo4j Cypher syntax.\n\n"
                f"Question: {question}"
            ),
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    # Keep only the first generated segment if the model starts
    # adding another question, answer, or Markdown block.
    for marker in (
        "\n```",
        "\nQuestion:",
        "\nGraph schema:",
        "\nCypher:",
        "\nAnswer:",
    ):
        if marker in text:
            text = text.split(marker, 1)[0].strip()

    # Remove a leading Markdown fence if present.
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].strip()

    return text

test_question = eval_rows[0]["question"]
print("Question:", test_question)
print("Generated query:\n", generate_cypher_clean(model, test_question))


In [ ]:

import csv
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/content/querycraft_dataset")
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ROOT / "eval.csv", encoding="utf-8-sig", newline="") as f:
    eval_rows = list(csv.DictReader(f))

cleaned_predictions = []

for row in eval_rows:
    query = generate_cypher_clean(model, row["question"])

    cleaned_predictions.append({
        "id": row["id"],
        "question": row["question"],
        "expected_cypher": row["cypher"],
        "predicted_cypher": query,
        "expected_result": row["expected_result"],
        "prediction_status": "GENERATED" if query else "EMPTY_OUTPUT",
    })

CLEAN_FILE = ARTIFACT_DIR / "finetuned_predictions_cleaned.csv"

with open(CLEAN_FILE, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f, fieldnames=cleaned_predictions[0].keys()
    )
    writer.writeheader()
    writer.writerows(cleaned_predictions)

print("Saved:", CLEAN_FILE)
print("Generated predictions:", len(cleaned_predictions))

for row in cleaned_predictions:
    print("\n", row["id"])
    print(row["predicted_cypher"])


In [ ]:
PREDICTIONS_FILE = Path(
    "/content/drive/MyDrive/content/querycraft_artifacts/"
    "finetuned_predictions_cleaned.csv"
)

In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    print("=== NODE LABELS AND PROPERTY KEYS ===")

    records = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS property
        RETURN label, property, count(*) AS occurrences
        ORDER BY label, property
    """)

    for record in records:
        print(record.data())

    print("\n=== RELATIONSHIP TYPES AND PROPERTY KEYS ===")

    records = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship_type,
               keys(r) AS properties,
               count(*) AS occurrences
        ORDER BY relationship_type
    """)

    for record in records:
        print(record.data())

    print("\n=== SAMPLE NODES ===")

    records = session.run("""
        MATCH (n)
        RETURN labels(n) AS labels, properties(n) AS properties
        LIMIT 10
    """)

    for record in records:
        print(record.data())

In [ ]:
import pandas as pd

REPORT_FILE = (
    "/content/drive/MyDrive/content/"
    "querycraft_artifacts/validation_report.csv"
)

report_df = pd.read_csv(REPORT_FILE)

print(
    report_df.loc[
        report_df["status"] == "ERROR_OR_REJECTED",
        ["id", "question", "predicted_cypher", "error"]
    ].to_string(index=False)
)

In [ ]:

from pathlib import Path
import pandas as pd

EVAL_FILE = Path(
    "/content/drive/MyDrive/content/"
    "querycraft_dataset/eval.csv"
)

# Load evaluation dataset
eval_df = pd.read_csv(EVAL_FILE, dtype=str).fillna("")

# Confirm required columns exist
required_columns = {
    "id", "question", "cypher", "expected_result"
}
missing = required_columns - set(eval_df.columns)

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Back up the original file before modifying it
backup_file = EVAL_FILE.with_name("eval_backup_before_fixes.csv")

if not backup_file.exists():
    eval_df.to_csv(
        backup_file, index=False, encoding="utf-8-sig"
    )

# Corrected Cypher queries
corrected_queries = {
    "EV004": """MATCH ()-[r]->()
RETURN count(r) AS relationship_count""",

    "EV010": """MATCH (m:Movie)
WHERE NOT EXISTS {
    MATCH (m)-[:IN_GENRE]->(:Genre)
}
RETURN m.title AS title
ORDER BY title"""
}

# Update exactly one row for each ID
for record_id, query in corrected_queries.items():
    mask = eval_df["id"].eq(record_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected one {record_id} row; found {mask.sum()}"
        )

    eval_df.loc[mask, "cypher"] = query

# Save the corrected evaluation dataset
eval_df.to_csv(
    EVAL_FILE, index=False, encoding="utf-8-sig"
)

print("Updated evaluation file:", EVAL_FILE)
print("Original backup:", backup_file)


In [ ]:

import json

corrected_queries = {
    "EV004": """
        MATCH ()-[r]->()
        RETURN count(r) AS relationship_count
    """,
    "EV010": """
        MATCH (m:Movie)
        WHERE NOT EXISTS {
            MATCH (m)-[:IN_GENRE]->(:Genre)
        }
        RETURN m.title AS title
        ORDER BY title
    """
}

with driver.session(database=NEO4J_DATABASE) as session:
    for record_id, query in corrected_queries.items():
        rows = session.run(query).data()

        print(f"\n{record_id} result:")
        print(json.dumps(rows, indent=2, ensure_ascii=False))


In [ ]:

import json
import pandas as pd

EVAL_FILE = (
    "/content/drive/MyDrive/content/"
    "querycraft_dataset/eval.csv"
)

df = pd.read_csv(EVAL_FILE, dtype=str).fillna("")

verified_expected_results = {
    "EV004": [{"relationship_count": 3}],
    "EV010": []
}

for record_id, expected in verified_expected_results.items():
    mask = df["id"].eq(record_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one {record_id} row.")

    df.loc[mask, "expected_result"] = json.dumps(expected)

df.to_csv(EVAL_FILE, index=False, encoding="utf-8-sig")

print(
    df.loc[
        df["id"].isin(["EV004", "EV010"]),
        ["id", "question", "cypher", "expected_result"]
    ].to_string(index=False)
)


In [ ]:

import pandas as pd
from pathlib import Path

EVAL_FILE = Path(
    "/content/drive/MyDrive/content/"
    "querycraft_dataset/eval.csv"
)

PREDICTIONS_FILE = Path(
    "/content/drive/MyDrive/content/"
    "querycraft_artifacts/finetuned_predictions_corrected.csv"
)

eval_df = pd.read_csv(EVAL_FILE, dtype=str).fillna("")

prediction_rows = []

for _, row in eval_df.iterrows():
    predicted = generate_cypher_clean(
        model,
        row["question"],
        max_new_tokens=128
    )

    prediction_rows.append({
        "id": row["id"],
        "question": row["question"],
        "expected_cypher": row["cypher"],
        "predicted_cypher": predicted,
        "expected_result": row["expected_result"],
    })

    print(f"\n{row['id']}: {row['question']}")
    print("Generated Cypher:", predicted)

predictions_df = pd.DataFrame(prediction_rows)

PREDICTIONS_FILE.parent.mkdir(
    parents=True, exist_ok=True
)

predictions_df.to_csv(
    PREDICTIONS_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nPredictions saved to:", PREDICTIONS_FILE)


In [ ]:

import pandas as pd
from pathlib import Path

EVAL_FILE = Path(
    "/content/drive/MyDrive/content/"
    "querycraft_dataset/eval.csv"
)

PREDICTIONS_FILE = Path(
    "/content/drive/MyDrive/content/"
    "querycraft_artifacts/finetuned_predictions_corrected.csv"
)

eval_df = pd.read_csv(EVAL_FILE, dtype=str).fillna("")

prediction_rows = []

for _, row in eval_df.iterrows():
    predicted = generate_cypher_clean(
        model,
        row["question"],
        max_new_tokens=128
    )

    prediction_rows.append({
        "id": row["id"],
        "question": row["question"],
        "expected_cypher": row["cypher"],
        "predicted_cypher": predicted,
        "expected_result": row["expected_result"],
    })

    print(f"\n{row['id']}: {row['question']}")
    print("Generated Cypher:", predicted)

predictions_df = pd.DataFrame(prediction_rows)

PREDICTIONS_FILE.parent.mkdir(
    parents=True, exist_ok=True
)

predictions_df.to_csv(
    PREDICTIONS_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nPredictions saved to:", PREDICTIONS_FILE)


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        )
    )
else:
    raise RuntimeError(
        "A CUDA GPU is required for this QLoRA training setup."
    )

In [ ]:
%pip install -q -U "transformers>=5.10.1" "peft>=0.20.0" accelerate bitsandbytes datasets

In [ ]:

import os
import torch
import transformers
import peft
import bitsandbytes as bnb

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable")
if torch.cuda.is_available():
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)


In [ ]:

from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

search_roots = [
    Path("/content/querycraft_dataset"),
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

found_files = set()

for root in search_roots:
    if root.exists():
        for path in root.rglob("*.jsonl"):
            if "querycraft" in str(path).lower() or "reviewed" in path.name.lower():
                found_files.add(path)

print("Candidate JSONL files:")
for path in sorted(found_files, key=lambda p: str(p)):
    print(path)


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

SEARCH_ROOTS = [
    Path("/content/querycraft_dataset"),
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

dataset_files = set()

for root in SEARCH_ROOTS:
    if root.exists():
        for pattern in ("*.jsonl", "*.json", "*.csv", "*.zip"):
            for path in root.rglob(pattern):
                if "querycraft" in str(path).lower():
                    dataset_files.add(path)

print("=== QUERYCRAFT DATASET FILES FOUND ===")
for path in sorted(dataset_files, key=str):
    print(f"{path} | {path.stat().st_size:,} bytes")

if not dataset_files:
    print("No QueryCraft files found. Check your Drive folder or upload location.")


In [ ]:

import json

jsonl_files = sorted(
    [p for p in dataset_files if p.suffix.lower() == ".jsonl"],
    key=str
)

for path in jsonl_files:
    print(f"\n=== {path} ===")
    valid_count = 0
    invalid_count = 0
    split_counts = {}

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
                valid_count += 1

                split = str(
                    record.get("split", record.get("subset", "unspecified"))
                ).lower()
                split_counts[split] = split_counts.get(split, 0) + 1

            except json.JSONDecodeError as exc:
                invalid_count += 1
                print(f"Invalid JSON at line {line_number}: {exc}")

    print("Valid JSON records:", valid_count)
    print("Invalid JSON lines:", invalid_count)
    print("Split labels:", split_counts)


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

print("Searching Google Drive for train.jsonl and eval.jsonl...")

for path in MYDRIVE.rglob("*.jsonl"):
    if path.name.lower() in {"train.jsonl", "eval.jsonl", "reviewed.jsonl"}:
        print(path)


In [ ]:

from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

for filename in ["train.jsonl", "eval.jsonl", "reviewed.jsonl", "reviewed_train.jsonl"]:
    matches = list(MYDRIVE.rglob(filename))
    print(f"\n--- {filename} ---")
    if matches:
        for path in matches:
            print(path)
    else:
        print("Not found")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:

from pathlib import Path

for folder in [
    Path("/content/drive/MyDrive/content"),
    Path("/content/drive/MyDrive/content/querycraft_dataset"),
]:
    print(f"\n=== {folder} ===")
    if folder.exists():
        for item in sorted(folder.iterdir()):
            print("DIR " if item.is_dir() else "FILE", item.name)
    else:
        print("Folder does not exist")


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

print("Searching Google Drive for train.jsonl and evaluation.jsonl...")

for path in MYDRIVE.rglob("*.jsonl"):
    if path.name.lower() in {"train.jsonl", "evaluation.jsonl", "reviewed.jsonl"}:
        print(path)


In [ ]:

import json
from pathlib import Path

# CHANGE THIS to the actual directory containing train.jsonl and eval.jsonl.
DATASET_DIR = Path("/content/drive/MyDrive/content/querycraft_dataset")

TRAIN_PATH = DATASET_DIR / "train.jsonl"
EVAL_PATH = DATASET_DIR / "evaluation.jsonl"

assert TRAIN_PATH.is_file(), f"Training file not found: {TRAIN_PATH}"
assert EVAL_PATH.is_file(), f"Evaluation file not found: {EVAL_PATH}"

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_no}: {exc}"
                ) from exc
    return records

train_records = load_jsonl(TRAIN_PATH)
eval_records = load_jsonl(EVAL_PATH)

print("Training records:", len(train_records))
print("Evaluation records:", len(eval_records))
print("Combined records:", len(train_records) + len(eval_records))

assert len(train_records) + len(eval_records) == 40, (
    "The combined count is not 40. Check the selected files before continuing."
)

print("Both files contain valid JSON records.")


In [ ]:

def get_question(record):
    for key in ("instruction", "question", "prompt", "natural_language"):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

def get_cypher(record):
    for key in ("cypher", "query", "response", "output", "target"):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

def pair_key(record):
    question = get_question(record)
    cypher = get_cypher(record)

    if question is None or cypher is None:
        raise ValueError(
            f"Missing question or Cypher field. Available fields: {list(record)}"
        )

    return (question, cypher)

train_keys = [pair_key(r) for r in train_records]
eval_keys = [pair_key(r) for r in eval_records]

cross_split_duplicates = set(train_keys) & set(eval_keys)

print("Duplicate question–Cypher pairs across splits:",
      len(cross_split_duplicates))

if cross_split_duplicates:
    for question, cypher in cross_split_duplicates:
        print("\nQuestion:", question)
        print("Cypher:", cypher)

    raise ValueError(
        "Duplicates detected across train/eval. Review the records before combining."
    )

print("Cross-split duplicate check passed.")


In [ ]:

OUTPUT_DIR = Path("/content/drive/MyDrive/content")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "reviewed.jsonl"

combined_records = train_records + eval_records

# Refuse to overwrite an existing reviewed.jsonl file.
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"{OUTPUT_PATH} already exists. Rename or back it up before creating a new file."
    )

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in combined_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Created:", OUTPUT_PATH)
print("Records written:", len(combined_records))


In [ ]:

verified_records = load_jsonl(OUTPUT_PATH)

print("Output file:", OUTPUT_PATH)
print("Verified record count:", len(verified_records))
print("First record fields:",
      list(verified_records[0].keys()) if verified_records else "EMPTY")
print("Last record fields:",
      list(verified_records[-1].keys()) if verified_records else "EMPTY")

assert len(verified_records) == 40
assert all(get_question(r) and get_cypher(r) for r in verified_records)

print("\nSUCCESS: reviewed.jsonl contains exactly 40 valid records.")


In [ ]:

assert "driver" in globals(), "Reconnect using your existing Neo4j credentials first."
assert "NEO4J_DATABASE" in globals(), "Set your existing Neo4j database name."

with driver.session(database=NEO4J_DATABASE) as session:
    node_info = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS property
        RETURN DISTINCT label, property
        ORDER BY label, property
    """).data()

    relationship_info = session.run("""
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS source_labels,
               type(r) AS relationship_type,
               labels(b) AS target_labels
        ORDER BY relationship_type
    """).data()

print("NODE LABELS AND PROPERTIES")
for item in node_info:
    print(item)

print("\nRELATIONSHIP PATTERNS")
for item in relationship_info:
    print(item)


In [ ]:

from google.colab import userdata
from neo4j import GraphDatabase

# Read existing credentials from Colab Secrets.
NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

# Use your existing database name if stored as a Colab Secret.
try:
    NEO4J_DATABASE = userdata.get("NEO4J_DATABASE")
except Exception:
    NEO4J_DATABASE = None

# AuraDB's usual default database name is "neo4j".
if not NEO4J_DATABASE:
    NEO4J_DATABASE = "neo4j"

assert NEO4J_URI, "NEO4J_URI is missing from Colab Secrets."
assert NEO4J_USERNAME, "NEO4J_USERNAME is missing from Colab Secrets."
assert NEO4J_PASSWORD, "NEO4J_PASSWORD is missing from Colab Secrets."

# Close an existing driver if this cell is rerun.
if "driver" in globals():
    try:
        driver.close()
    except Exception:
        pass

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    connection_timeout=30,
)

driver.verify_connectivity()

print("Neo4j connection successful.")
print("Database selected:", NEO4J_DATABASE)


In [ ]:

assert "driver" in globals(), "Run Step 1 to connect to Neo4j."
assert "NEO4J_DATABASE" in globals(), "Database name is not set."

with driver.session(database=NEO4J_DATABASE) as session:

    print("=== NODE LABELS ===")
    labels = session.run("""
        CALL db.labels()
        YIELD label
        RETURN label
        ORDER BY label
    """).value()
    print(labels or "No node labels found.")

    print("\n=== RELATIONSHIP TYPES ===")
    relationship_types = session.run("""
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType
        ORDER BY relationshipType
    """).value()
    print(relationship_types or "No relationship types found.")

    print("\n=== NODE PROPERTIES ===")
    node_info = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS property
        RETURN DISTINCT label, property
        ORDER BY label, property
    """).data()

    for item in node_info:
        print(item)

    print("\n=== RELATIONSHIP PATTERNS ===")
    relationship_info = session.run("""
        MATCH (a)-[r]->(b)
        RETURN DISTINCT
            labels(a) AS source_labels,
            type(r) AS relationship_type,
            labels(b) AS target_labels
        ORDER BY relationship_type
    """).data()

    for item in relationship_info:
        print(item)

    if not relationship_info:
        print("No existing relationships found.")


In [ ]:

import hashlib
import random

def first_nonempty(record, candidates):
    for key in candidates:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

examples = []

for i, record in enumerate(records):
    question = first_nonempty(record, [
        "instruction", "question", "prompt", "natural_language",
        "natural_language_question", "input"
    ])
    cypher = first_nonempty(record, [
        "cypher", "query", "target", "response", "output", "expected_cypher"
    ])

    if not question or not cypher:
        raise ValueError(
            f"Record {i} has missing question/Cypher. "
            f"Available fields: {list(record.keys())}"
        )

    examples.append({"question": question, "cypher": cypher})

def fingerprint(example):
    content = example["question"].strip() + "\n" + example["cypher"].strip()
    return hashlib.sha256(content.encode("utf-8")).hexdigest()

assert len({fingerprint(x) for x in examples}) == len(examples), (
    "Duplicate question/Cypher pairs detected. Review before training."
)

print("Usable examples:", len(examples))
for item in examples[:3]:
    print("\nQUESTION:", item["question"])
    print("CYPHER:", item["cypher"])


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

matches = list(MYDRIVE.rglob("reviewed.jsonl"))

print("Reviewed JSONL files found:", len(matches))
for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")


In [ ]:

import json
from pathlib import Path

# Replace with the exact path printed in Step 1.
DATASET_PATH = "/content/drive/MyDrive/content/reviewed.jsonl"

path = Path(DATASET_PATH)
assert path.is_file(), f"Dataset file not found: {path}"

records = []

with open(path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON on line {line_no}: {exc}"
            ) from exc

print("Dataset loaded:", path)
print("Total records:", len(records))

assert len(records) == 40, (
    f"Expected 40 records, but found {len(records)}. "
    "Check that you selected the correct reviewed.jsonl file."
)

print("Record fields:", list(records[0].keys()))
print("Dataset loading successful.")


In [ ]:

random.seed(42)

shuffled = examples.copy()
random.shuffle(shuffled)

split_at = int(len(shuffled) * 0.8)
train_examples = shuffled[:split_at]
val_examples = shuffled[split_at:]

assert not set(map(fingerprint, train_examples)) & set(
    map(fingerprint, val_examples)
)

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))


In [ ]:

schema_lines = []

if node_info:
    schema_lines.append("Node properties:")
    for item in node_info:
        schema_lines.append(f"- ({item['label']}.{item['property']})")

if relationship_info:
    schema_lines.append("Relationship patterns:")
    for item in relationship_info:
        source = ":".join(item["source_labels"]) or "Unknown"
        target = ":".join(item["target_labels"]) or "Unknown"
        schema_lines.append(
            f"- ({source})-[:{item['relationship_type']}]->({target})"
        )

SCHEMA_TEXT = "\n".join(schema_lines)

if not SCHEMA_TEXT.strip():
    raise ValueError(
        "No schema details found. Verify the existing database schema "
        "or provide the schema from your original QueryCraft setup."
    )

print(SCHEMA_TEXT)


In [ ]:

schema_lines = []

if node_info:
    schema_lines.append("Node properties:")
    for item in node_info:
        schema_lines.append(f"- ({item['label']}.{item['property']})")

if relationship_info:
    schema_lines.append("Relationship patterns:")
    for item in relationship_info:
        source = ":".join(item["source_labels"]) or "Unknown"
        target = ":".join(item["target_labels"]) or "Unknown"
        schema_lines.append(
            f"- ({source})-[:{item['relationship_type']}]->({target})"
        )

SCHEMA_TEXT = "\n".join(schema_lines)

if not SCHEMA_TEXT.strip():
    raise ValueError(
        "No schema details found. Verify the existing database schema "
        "or provide the schema from your original QueryCraft setup."
    )

print(SCHEMA_TEXT)


In [ ]:

from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType,
)

print("PEFT functions imported successfully.")


In [ ]:

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("PEFT functions imported successfully.")


In [ ]:

import torch

assert "model" in globals(), (
    "Model is not loaded. Run your Gemma 3 4B IT model-loading cell first."
)

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable the NVIDIA T4 GPU runtime."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Model class:", type(model).__name__)
print("4-bit loaded:", getattr(model, "is_loaded_in_4bit", False))


In [ ]:

import hashlib
import random

def first_nonempty(record, candidates):
    for key in candidates:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

examples = []

for i, record in enumerate(records):
    question = first_nonempty(record, [
        "instruction", "question", "prompt", "natural_language",
        "natural_language_question", "input"
    ])
    cypher = first_nonempty(record, [
        "cypher", "query", "target", "response", "output", "expected_cypher"
    ])

    if not question or not cypher:
        raise ValueError(
            f"Record {i} has missing question/Cypher. "
            f"Available fields: {list(record.keys())}"
        )

    examples.append({"question": question, "cypher": cypher})

def fingerprint(example):
    content = example["question"].strip() + "\n" + example["cypher"].strip()
    return hashlib.sha256(content.encode("utf-8")).hexdigest()

assert len({fingerprint(x) for x in examples}) == len(examples), (
    "Duplicate question/Cypher pairs detected. Review before training."
)

print("Usable examples:", len(examples))
for item in examples[:3]:
    print("\nQUESTION:", item["question"])
    print("CYPHER:", item["cypher"])


In [ ]:

random.seed(42)

shuffled = examples.copy()
random.shuffle(shuffled)

split_at = int(len(shuffled) * 0.8)
train_examples = shuffled[:split_at]
val_examples = shuffled[split_at:]

assert not set(map(fingerprint, train_examples)) & set(
    map(fingerprint, val_examples)
)

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))


In [ ]:

schema_lines = []

if node_info:
    schema_lines.append("Node properties:")
    for item in node_info:
        schema_lines.append(f"- ({item['label']}.{item['property']})")

if relationship_info:
    schema_lines.append("Relationship patterns:")
    for item in relationship_info:
        source = ":".join(item["source_labels"]) or "Unknown"
        target = ":".join(item["target_labels"]) or "Unknown"
        schema_lines.append(
            f"- ({source})-[:{item['relationship_type']}]->({target})"
        )

SCHEMA_TEXT = "\n".join(schema_lines)

if not SCHEMA_TEXT.strip():
    raise ValueError(
        "No schema details found. Verify the existing database schema "
        "or provide the schema from your original QueryCraft setup."
    )

print(SCHEMA_TEXT)


In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    Gemma3ForConditionalGeneration,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

MODEL_ID = "google/gemma-3-4b-it"

assert torch.cuda.is_available(), "Enable the Colab GPU first."

compute_dtype = torch.float16  # Suitable starting point for NVIDIA T4.

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    attn_implementation="eager",
)

model.config.use_cache = False

print("Model loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Quantized:", getattr(model, "is_loaded_in_4bit", "Check model configuration"))


In [ ]:

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

assert any(p.requires_grad for p in model.parameters()), (
    "No trainable parameters found."
)


In [ ]:

MAX_LENGTH = 1024

def make_prompt(question):
    messages = [
        {
            "role": "system",
            "content": (
                "You generate Cypher queries for the QueryCraft graph database. "
                "Use only the supplied schema. Do not invent labels, properties, "
                "or relationship types. Return only one read-only Cypher query. "
                "Never generate CREATE, MERGE, SET, DELETE, REMOVE, DROP, "
                "LOAD CSV, or write procedures."
                "\n\nGRAPH SCHEMA:\n" + SCHEMA_TEXT
            ),
        },
        {
            "role": "user",
            "content": "Translate this question into Cypher:\n" + question,
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def encode_example(example):
    prompt = make_prompt(example["question"])
    answer = example["cypher"].strip()

    # Preserve the original reviewed target; add the EOS token for training.
    full_text = prompt + answer + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"]

    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    prompt_length = len(prompt_ids)

    if prompt_length >= len(input_ids):
        raise ValueError(
            "Answer was truncated or missing. Reduce prompt size or increase "
            "MAX_LENGTH after checking GPU memory."
        )

    labels = input_ids.copy()
    labels[:min(prompt_length, len(labels))] = [
        -100
    ] * min(prompt_length, len(labels))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_data = [encode_example(x) for x in train_examples]
val_data = [encode_example(x) for x in val_examples]

print("Encoded training samples:", len(train_data))
print("Encoded validation samples:", len(val_data))
print("Example token length:", len(train_data[0]["input_ids"]))


In [ ]:

from torch.utils.data import Dataset, DataLoader

class CypherDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

def collate_batch(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id

    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_id] * pad_len)
        attention_masks.append(item["attention_mask"] + [0] * pad_len)
        labels.append(item["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

BATCH_SIZE = 1
GRAD_ACCUM = 8

train_loader = DataLoader(
    CypherDataset(train_data),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    CypherDataset(val_data),
    batch_size=1,
    shuffle=False,
    collate_fn=collate_batch,
)

print("Training batches per epoch:", len(train_loader))
print("Optimizer updates per epoch:", (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM)


In [ ]:

from torch.optim import AdamW
from pathlib import Path
import math
import json
import time

OUTPUT_DIR = Path("/content/drive/MyDrive/QueryCraft_Gemma3_4B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 3
LEARNING_RATE = 1e-4

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

def evaluate_loss():
    model.eval()
    total_loss = 0.0
    batches = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to("cuda:0") for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)

            loss = outputs.loss
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite validation loss detected.")

            total_loss += loss.item()
            batches += 1

    model.train()
    return total_loss / max(batches, 1)

history = []
optimizer.zero_grad(set_to_none=True)
model.train()
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = {k: v.to("cuda:0") for k, v in batch.items()}

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError(
                "Non-finite training loss. Stop and inspect numerical stability."
            )

        (loss / GRAD_ACCUM).backward()
        epoch_loss += loss.item()

        is_update_step = (
            (step + 1) % GRAD_ACCUM == 0
            or (step + 1) == len(train_loader)
        )

        if is_update_step:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if (step + 1) % 8 == 0 or (step + 1) == len(train_loader):
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Batch {step+1}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = epoch_loss / len(train_loader)
    val_loss = evaluate_loss()

    entry = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss,
    }
    history.append(entry)

    print(
        f"Epoch {epoch+1} complete | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f}"
    )

    model.save_pretrained(OUTPUT_DIR / f"checkpoint_epoch_{epoch+1}")
    tokenizer.save_pretrained(OUTPUT_DIR / f"checkpoint_epoch_{epoch+1}")

with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Training time (minutes):", round((time.time() - start_time) / 60, 2))
print("Checkpoint directory:", OUTPUT_DIR)


In [ ]:

FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

# Save a copy of the exact reviewed dataset used by this run.
with open(OUTPUT_DIR / "reviewed_dataset_used.jsonl", "w", encoding="utf-8") as f:
    for example in examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "base_model": MODEL_ID,
        "quantization": "4-bit NF4 double quantization",
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "train_examples": len(train_examples),
        "validation_examples": len(val_examples),
        "max_length": MAX_LENGTH,
    }, f, indent=2)

print("Final adapter:", FINAL_ADAPTER_DIR)
print("Adapter files:")
for path in sorted(FINAL_ADAPTER_DIR.iterdir()):
    print(path.name)


In [ ]:

def generate_cypher(question, max_new_tokens=160):
    model.eval()
    prompt = make_prompt(question)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda:0")

    previous_use_cache = model.config.use_cache
    model.config.use_cache = True

    try:
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    finally:
        model.config.use_cache = previous_use_cache

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_question = val_examples[0]["question"]

print("QUESTION:")
print(test_question)

print("\nEXPECTED CYPHER:")
print(val_examples[0]["cypher"])

print("\nGENERATED CYPHER:")
generated_query = generate_cypher(test_question)
print(generated_query)


In [ ]:

import re

def clean_generated_query(text):
    text = text.strip()
    text = re.sub(r"^```(?:cypher)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

FORBIDDEN = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|"
    r"GRANT|DENY|REVOKE|LOAD|CALL|FOREACH|TERMINATE|USE)\b",
    flags=re.IGNORECASE,
)

ALLOWED_START = re.compile(
    r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH|UNWIND|RETURN)\b",
    flags=re.IGNORECASE,
)

def check_readonly_cypher(query):
    query = clean_generated_query(query)

    if not query:
        return False, "Empty query"

    if ";" in query:
        return False, "Semicolons/multiple statements are not allowed"

    if "--" in query or "/*" in query or "*/" in query:
        return False, "Comments are not allowed"

    if FORBIDDEN.search(query):
        return False, "Forbidden write clause or procedure detected"

    if not ALLOWED_START.search(query):
        return False, "Query does not begin with an allowed read clause"

    if not re.search(r"\bRETURN\b", query, flags=re.IGNORECASE):
        return False, "Query must return results"

    # Conservative policy: reject queries that invoke functions or
    # constructs requiring separate security review.
    if re.search(r"\b(apoc|gds)\s*\.", query, flags=re.IGNORECASE):
        return False, "External procedure/function namespace not allowed"

    return True, "Passed lexical safety checks"


In [ ]:

from neo4j import READ_ACCESS

def evaluate_readonly_query(query):
    ok, reason = check_readonly_cypher(query)
    if not ok:
        return {"safe": False, "reason": reason}

    # Cap returned rows. Existing LIMIT clauses are preserved.
    query_to_run = query
    if not re.search(r"\bLIMIT\b", query, flags=re.IGNORECASE):
        query_to_run = query.rstrip() + "\nLIMIT 10"

    def read_work(tx):
        # EXPLAIN checks the query plan without executing the data query.
        tx.run("EXPLAIN " + query_to_run).consume()

        # Actual execution occurs only within an explicit read transaction.
        result = tx.run(query_to_run)
        rows = []
        for record in result:
            rows.append(record.data())
            if len(rows) >= 10:
                break

        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            rows = session.execute_read(read_work, timeout=5)

        return {
            "safe": True,
            "reason": "Query executed successfully",
            "rows_returned": len(rows),
            "sample_rows": rows[:3],
        }

    except Exception as exc:
        return {
            "safe": True,
            "reason": "Neo4j rejected the query or execution failed",
            "error": str(exc)[:500],
        }

print(evaluate_readonly_query(generated_query))


In [ ]:

evaluation_results = []

for i, example in enumerate(val_examples, start=1):
    prediction = generate_cypher(example["question"])
    execution = evaluate_readonly_query(prediction)

    result = {
        "question": example["question"],
        "expected_cypher": example["cypher"],
        "generated_cypher": prediction,
        "safety_check": check_readonly_cypher(prediction),
        "neo4j_result": execution,
    }
    evaluation_results.append(result)

    print(f"\n--- Validation example {i}/{len(val_examples)} ---")
    print("Question:", example["question"])
    print("Generated:", prediction)
    print("Safety:", result["safety_check"])
    print("Neo4j:", execution)

report_path = OUTPUT_DIR / "validation_results.jsonl"

with open(report_path, "w", encoding="utf-8") as f:
    for result in evaluation_results:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")

passed_safety = sum(
    r["safety_check"][0] for r in evaluation_results
)
executed_successfully = sum(
    r["neo4j_result"].get("reason") == "Query executed successfully"
    for r in evaluation_results
)

print("\n=== EVALUATION SUMMARY ===")
print("Validation examples:", len(evaluation_results))
print("Passed safety checks:", passed_safety)
print("Executed successfully:", executed_successfully)
print("Saved report:", report_path)


In [ ]:

import torch
import re
import json
from pathlib import Path

assert "model" in globals(), "Load your trained Gemma/LoRA model first."
assert "driver" in globals(), "Reconnect to Neo4j first."
assert "NEO4J_DATABASE" in globals(), "Set your Neo4j database name."
assert "generate_cypher" in globals(), "Run your query-generation function first."

with driver.session(database=NEO4J_DATABASE) as session:
    labels = session.run("""
        CALL db.labels()
        YIELD label
        RETURN label ORDER BY label
    """).value()

    relationship_types = session.run("""
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType ORDER BY relationshipType
    """).value()

print("Database:", NEO4J_DATABASE)
print("Node labels:", labels)
print("Relationship types:", relationship_types)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:

unseen_tests = [
    {
        "id": "UQ01",
        "question": "How many movie nodes are stored in the graph?",
        "reference_cypher": """
            MATCH (m:Movie)
            RETURN count(m) AS answer
        """,
    },
    {
        "id": "UQ02",
        "question": "What is the total number of people represented in the database?",
        "reference_cypher": """
            MATCH (p:Person)
            RETURN count(p) AS answer
        """,
    },
    {
        "id": "UQ03",
        "question": "Count all the genre nodes available in the graph.",
        "reference_cypher": """
            MATCH (g:Genre)
            RETURN count(g) AS answer
        """,
    },
    {
        "id": "UQ04",
        "question": "How many director-to-movie relationships exist?",
        "reference_cypher": """
            MATCH (:Person)-[r:DIRECTED]->(:Movie)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ05",
        "question": "How many acting credits connect people to movies?",
        "reference_cypher": """
            MATCH (:Person)-[r:ACTED_IN]->(:Movie)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ06",
        "question": "Count the movie-to-genre links in the database.",
        "reference_cypher": """
            MATCH (:Movie)-[r:IN_GENRE]->(:Genre)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ07",
        "question": "How many distinct movies have at least one director?",
        "reference_cypher": """
            MATCH (:Person)-[:DIRECTED]->(m:Movie)
            RETURN count(DISTINCT m) AS answer
        """,
    },
    {
        "id": "UQ08",
        "question": "How many distinct people have acted in at least one movie?",
        "reference_cypher": """
            MATCH (p:Person)-[:ACTED_IN]->(:Movie)
            RETURN count(DISTINCT p) AS answer
        """,
    },
]

print("New test questions:", len(unseen_tests))
for test in unseen_tests:
    print(test["id"], test["question"])


In [ ]:

existing_questions = set()

for variable_name in ("examples", "train_examples", "val_examples"):
    items = globals().get(variable_name, [])
    for item in items:
        question = item.get("question") or item.get("instruction") or item.get("prompt")
        if isinstance(question, str):
            existing_questions.add(" ".join(question.lower().split()))

overlap = [
    test for test in unseen_tests
    if " ".join(test["question"].lower().split()) in existing_questions
]

if overlap:
    print("Exact overlap detected:")
    for test in overlap:
        print(test["id"], test["question"])
    raise ValueError("Replace overlapping questions before evaluation.")

print("No exact question overlap found in the loaded dataset variables.")
print("Also review the questions manually for paraphrases of training examples.")


In [ ]:

from neo4j import READ_ACCESS

FORBIDDEN_CYPHER = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|"
    r"GRANT|DENY|REVOKE|LOAD|CALL|FOREACH|TERMINATE|USE|"
    r"APOC|GDS)\b",
    re.IGNORECASE,
)

ALLOWED_QUERY_START = re.compile(
    r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH|UNWIND)\b",
    re.IGNORECASE,
)

def safety_check(query):
    query = query.strip()

    # Remove a surrounding Markdown code fence, if present.
    query = re.sub(
        r"^```(?:cypher)?\s*|\s*```$",
        "",
        query,
        flags=re.IGNORECASE,
    ).strip()

    if not query:
        return False, "Empty query"

    if ";" in query:
        return False, "Semicolons or multiple statements are not allowed"

    if "--" in query or "/*" in query or "*/" in query:
        return False, "Comments are not allowed"

    if FORBIDDEN_CYPHER.search(query):
        return False, "Forbidden clause or procedure detected"

    if not ALLOWED_QUERY_START.search(query):
        return False, "Query does not start with an allowed read clause"

    if not re.search(r"\bRETURN\b", query, re.IGNORECASE):
        return False, "RETURN clause is required"

    return True, "Passed lexical checks"


def execute_readonly(query, timeout_seconds=5):
    passed, reason = safety_check(query)

    if not passed:
        return {
            "success": False,
            "reason": reason,
            "rows": None,
        }

    query = query.strip()

    def read_work(tx):
        # EXPLAIN checks whether Neo4j can plan the query without
        # executing its data retrieval.
        tx.run("EXPLAIN " + query).consume()

        result = tx.run(query)
        rows = []
        for record in result:
            rows.append(record.data())
            if len(rows) > 100:
                raise ValueError("Result exceeded the 100-row evaluation limit.")
        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            rows = session.execute_read(
                read_work,
                timeout=timeout_seconds,
            )

        return {
            "success": True,
            "reason": "Query executed successfully",
            "rows": rows,
        }

    except Exception as exc:
        return {
            "success": False,
            "reason": str(exc)[:500],
            "rows": None,
        }


def canonical_values(rows):
    """Normalize rows for alias-independent comparison."""
    if rows is None:
        return None

    normalized = [
        sorted(row.values(), key=lambda value: json.dumps(
            value, sort_keys=True, default=str
        ))
        for row in rows
    ]

    return sorted(
        normalized,
        key=lambda row: json.dumps(row, sort_keys=True, default=str),
    )

print("Read-only evaluation helpers are ready.")


In [ ]:

existing_questions = set()

for variable_name in ("examples", "train_examples", "val_examples"):
    items = globals().get(variable_name, [])
    for item in items:
        question = item.get("question") or item.get("instruction") or item.get("prompt")
        if isinstance(question, str):
            existing_questions.add(" ".join(question.lower().split()))

overlap = [
    test for test in unseen_tests
    if " ".join(test["question"].lower().split()) in existing_questions
]

if overlap:
    print("Exact overlap detected:")
    for test in overlap:
        print(test["id"], test["question"])
    raise ValueError("Replace overlapping questions before evaluation.")

print("No exact question overlap found in the loaded dataset variables.")
print("Also review the questions manually for paraphrases of training examples.")


In [ ]:

from neo4j import READ_ACCESS

FORBIDDEN_CYPHER = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|"
    r"GRANT|DENY|REVOKE|LOAD|CALL|FOREACH|TERMINATE|USE|"
    r"APOC|GDS)\b",
    re.IGNORECASE,
)

ALLOWED_QUERY_START = re.compile(
    r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH|UNWIND)\b",
    re.IGNORECASE,
)

def safety_check(query):
    query = query.strip()

    # Remove a surrounding Markdown code fence, if present.
    query = re.sub(
        r"^```(?:cypher)?\s*|\s*```$",
        "",
        query,
        flags=re.IGNORECASE,
    ).strip()

    if not query:
        return False, "Empty query"

    if ";" in query:
        return False, "Semicolons or multiple statements are not allowed"

    if "--" in query or "/*" in query or "*/" in query:
        return False, "Comments are not allowed"

    if FORBIDDEN_CYPHER.search(query):
        return False, "Forbidden clause or procedure detected"

    if not ALLOWED_QUERY_START.search(query):
        return False, "Query does not start with an allowed read clause"

    if not re.search(r"\bRETURN\b", query, re.IGNORECASE):
        return False, "RETURN clause is required"

    return True, "Passed lexical checks"


def execute_readonly(query, timeout_seconds=5):
    passed, reason = safety_check(query)

    if not passed:
        return {
            "success": False,
            "reason": reason,
            "rows": None,
        }

    query = query.strip()

    def read_work(tx):
        # EXPLAIN checks whether Neo4j can plan the query without
        # executing its data retrieval.
        tx.run("EXPLAIN " + query).consume()

        result = tx.run(query)
        rows = []
        for record in result:
            rows.append(record.data())
            if len(rows) > 100:
                raise ValueError("Result exceeded the 100-row evaluation limit.")
        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            rows = session.execute_read(
                read_work,
                timeout=timeout_seconds,
            )

        return {
            "success": True,
            "reason": "Query executed successfully",
            "rows": rows,
        }

    except Exception as exc:
        return {
            "success": False,
            "reason": str(exc)[:500],
            "rows": None,
        }


def canonical_values(rows):
    """Normalize rows for alias-independent comparison."""
    if rows is None:
        return None

    normalized = [
        sorted(row.values(), key=lambda value: json.dumps(
            value, sort_keys=True, default=str
        ))
        for row in rows
    ]

    return sorted(
        normalized,
        key=lambda row: json.dumps(row, sort_keys=True, default=str),
    )

print("Read-only evaluation helpers are ready.")


In [ ]:

import torch
import re
import json
from pathlib import Path

assert "model" in globals(), "Load your trained Gemma/LoRA model first."
assert "driver" in globals(), "Reconnect to Neo4j first."
assert "NEO4J_DATABASE" in globals(), "Set your Neo4j database name."
assert "generate_cypher" in globals(), "Run your query-generation function first."

with driver.session(database=NEO4J_DATABASE) as session:
    labels = session.run("""
        CALL db.labels()
        YIELD label
        RETURN label ORDER BY label
    """).value()

    relationship_types = session.run("""
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType ORDER BY relationshipType
    """).value()

print("Database:", NEO4J_DATABASE)
print("Node labels:", labels)
print("Relationship types:", relationship_types)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:

unseen_tests = [
    {
        "id": "UQ01",
        "question": "How many movie nodes are stored in the graph?",
        "reference_cypher": """
            MATCH (m:Movie)
            RETURN count(m) AS answer
        """,
    },
    {
        "id": "UQ02",
        "question": "What is the total number of people represented in the database?",
        "reference_cypher": """
            MATCH (p:Person)
            RETURN count(p) AS answer
        """,
    },
    {
        "id": "UQ03",
        "question": "Count all the genre nodes available in the graph.",
        "reference_cypher": """
            MATCH (g:Genre)
            RETURN count(g) AS answer
        """,
    },
    {
        "id": "UQ04",
        "question": "How many director-to-movie relationships exist?",
        "reference_cypher": """
            MATCH (:Person)-[r:DIRECTED]->(:Movie)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ05",
        "question": "How many acting credits connect people to movies?",
        "reference_cypher": """
            MATCH (:Person)-[r:ACTED_IN]->(:Movie)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ06",
        "question": "Count the movie-to-genre links in the database.",
        "reference_cypher": """
            MATCH (:Movie)-[r:IN_GENRE]->(:Genre)
            RETURN count(r) AS answer
        """,
    },
    {
        "id": "UQ07",
        "question": "How many distinct movies have at least one director?",
        "reference_cypher": """
            MATCH (:Person)-[:DIRECTED]->(m:Movie)
            RETURN count(DISTINCT m) AS answer
        """,
    },
    {
        "id": "UQ08",
        "question": "How many distinct people have acted in at least one movie?",
        "reference_cypher": """
            MATCH (p:Person)-[:ACTED_IN]->(:Movie)
            RETURN count(DISTINCT p) AS answer
        """,
    },
]

print("New test questions:", len(unseen_tests))
for test in unseen_tests:
    print(test["id"], test["question"])


In [ ]:

existing_questions = set()

for variable_name in ("examples", "train_examples", "val_examples"):
    items = globals().get(variable_name, [])
    for item in items:
        question = item.get("question") or item.get("instruction") or item.get("prompt")
        if isinstance(question, str):
            existing_questions.add(" ".join(question.lower().split()))

overlap = [
    test for test in unseen_tests
    if " ".join(test["question"].lower().split()) in existing_questions
]

if overlap:
    print("Exact overlap detected:")
    for test in overlap:
        print(test["id"], test["question"])
    raise ValueError("Replace overlapping questions before evaluation.")

print("No exact question overlap found in the loaded dataset variables.")
print("Also review the questions manually for paraphrases of training examples.")


In [ ]:

from neo4j import READ_ACCESS

FORBIDDEN_CYPHER = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|"
    r"GRANT|DENY|REVOKE|LOAD|CALL|FOREACH|TERMINATE|USE|"
    r"APOC|GDS)\b",
    re.IGNORECASE,
)

ALLOWED_QUERY_START = re.compile(
    r"^\s*(MATCH|OPTIONAL\s+MATCH|WITH|UNWIND)\b",
    re.IGNORECASE,
)

def safety_check(query):
    query = query.strip()

    # Remove a surrounding Markdown code fence, if present.
    query = re.sub(
        r"^```(?:cypher)?\s*|\s*```$",
        "",
        query,
        flags=re.IGNORECASE,
    ).strip()

    if not query:
        return False, "Empty query"

    if ";" in query:
        return False, "Semicolons or multiple statements are not allowed"

    if "--" in query or "/*" in query or "*/" in query:
        return False, "Comments are not allowed"

    if FORBIDDEN_CYPHER.search(query):
        return False, "Forbidden clause or procedure detected"

    if not ALLOWED_QUERY_START.search(query):
        return False, "Query does not start with an allowed read clause"

    if not re.search(r"\bRETURN\b", query, re.IGNORECASE):
        return False, "RETURN clause is required"

    return True, "Passed lexical checks"


def execute_readonly(query, timeout_seconds=5):
    passed, reason = safety_check(query)

    if not passed:
        return {
            "success": False,
            "reason": reason,
            "rows": None,
        }

    query = query.strip()

    def read_work(tx):
        # EXPLAIN checks whether Neo4j can plan the query without
        # executing its data retrieval.
        tx.run("EXPLAIN " + query).consume()

        result = tx.run(query)
        rows = []
        for record in result:
            rows.append(record.data())
            if len(rows) > 100:
                raise ValueError("Result exceeded the 100-row evaluation limit.")
        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            rows = session.execute_read(
                read_work,
                timeout=timeout_seconds,
            )

        return {
            "success": True,
            "reason": "Query executed successfully",
            "rows": rows,
        }

    except Exception as exc:
        return {
            "success": False,
            "reason": str(exc)[:500],
            "rows": None,
        }


def canonical_values(rows):
    """Normalize rows for alias-independent comparison."""
    if rows is None:
        return None

    normalized = [
        sorted(row.values(), key=lambda value: json.dumps(
            value, sort_keys=True, default=str
        ))
        for row in rows
    ]

    return sorted(
        normalized,
        key=lambda row: json.dumps(row, sort_keys=True, default=str),
    )

print("Read-only evaluation helpers are ready.")


In [ ]:

def clean_cypher_output(text):
    text = text.strip()
    text = re.sub(
        r"^```(?:cypher)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


evaluation_results = []

for test in unseen_tests:
    print("\n" + "=" * 70)
    print(test["id"], "-", test["question"])

    # Independent expected answer from the reviewed reference query.
    reference = execute_readonly(test["reference_cypher"])

    if not reference["success"]:
        result = {
            **test,
            "status": "REFERENCE_QUERY_FAILED",
            "reference_error": reference["reason"],
        }
        evaluation_results.append(result)
        print("Reference query failed:", reference["reason"])
        continue

    expected_rows = reference["rows"]

    # Model-generated query.
    try:
        generated = clean_cypher_output(
            generate_cypher(test["question"])
        )
    except Exception as exc:
        result = {
            **test,
            "status": "GENERATION_FAILED",
            "generation_error": str(exc)[:500],
            "expected_rows": expected_rows,
        }
        evaluation_results.append(result)
        print("Generation failed:", str(exc)[:300])
        continue

    model_result = execute_readonly(generated)

    if not model_result["success"]:
        result = {
            **test,
            "generated_cypher": generated,
            "status": "GENERATED_QUERY_FAILED",
            "failure_reason": model_result["reason"],
            "expected_rows": expected_rows,
        }
        evaluation_results.append(result)

        print("Expected result:", expected_rows)
        print("Generated Cypher:\n", generated)
        print("Generated query rejected or failed:", model_result["reason"])
        continue

    actual_rows = model_result["rows"]
    correct = (
        canonical_values(expected_rows)
        == canonical_values(actual_rows)
    )

    result = {
        **test,
        "generated_cypher": generated,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "status": "PASS" if correct else "WRONG_ANSWER",
    }
    evaluation_results.append(result)

    print("Expected result:", expected_rows)
    print("Actual result:", actual_rows)
    print("Generated Cypher:\n", generated)
    print("RESULT:", result["status"])


In [ ]:

from pathlib import Path
import json

REPORT_DIR = Path("/content/drive/MyDrive/QueryCraft_Gemma3_4B/unseen_evaluation")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

total = len(unseen_tests)
passed = sum(r.get("status") == "PASS" for r in evaluation_results)
wrong = sum(r.get("status") == "WRONG_ANSWER" for r in evaluation_results)
failed = sum(
    r.get("status") in {
        "GENERATED_QUERY_FAILED",
        "GENERATION_FAILED",
        "REFERENCE_QUERY_FAILED",
    }
    for r in evaluation_results
)

summary = {
    "model": "google/gemma-3-4b-it + QueryCraft LoRA adapter",
    "evaluation_type": "Unseen natural-language questions",
    "total_questions": total,
    "correct_answers": passed,
    "wrong_answers": wrong,
    "failed_tests": failed,
    "answer_accuracy_percent": round(100 * passed / total, 2) if total else 0,
    "evaluation_note": (
        "Accuracy is based on matching generated query results to "
        "independent reference query results on the current Neo4j database."
    ),
}

with open(REPORT_DIR / "unseen_evaluation_results.jsonl", "w", encoding="utf-8") as f:
    for item in evaluation_results:
        f.write(json.dumps(item, ensure_ascii=False, default=str) + "\n")

with open(REPORT_DIR / "unseen_evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n=== QUERYCRAFT UNSEEN EVALUATION SUMMARY ===")
print(json.dumps(summary, indent=2))
print("\nReports saved to:", REPORT_DIR)


In [ ]:

import torch

required_variables = [
    "model",
    "tokenizer",
    "driver",
    "NEO4J_DATABASE",
    "generate_cypher",
]

missing = [name for name in required_variables if name not in globals()]
assert not missing, (
    f"Missing variables: {missing}. Run the corresponding model-loading "
    "and Neo4j connection cells first."
)

assert torch.cuda.is_available(), "Enable the Colab GPU runtime."

driver.verify_connectivity()

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", type(model).__name__)
print("Neo4j database connection: OK")
print("QueryCraft is ready for Gradio.")


In [ ]:

assert "evaluation_results" in globals(), (
    "Run the unseen evaluation cell first."
)

for item in evaluation_results:
    print("\n" + "=" * 80)
    print("ID:", item.get("id"))
    print("Question:", item.get("question"))
    print("Status:", item.get("status"))

    print("\nExpected Cypher:")
    print(item.get("reference_cypher", item.get("expected_cypher", "Not recorded")))

    print("\nGenerated Cypher:")
    print(item.get("generated_cypher", "No query generated"))

    print("\nExpected results:")
    print(item.get("expected_rows", "Not recorded"))

    print("\nActual results:")
    print(item.get("actual_rows", "Not recorded"))

    print("\nFailure reason:")
    print(item.get("failure_reason",
                    item.get("generation_error",
                             item.get("reference_error", "None recorded"))))


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:
    patterns = session.run("""
        MATCH (a)-[r]->(b)
        RETURN DISTINCT
            labels(a) AS source_labels,
            type(r) AS relationship_type,
            labels(b) AS target_labels
        ORDER BY relationship_type
    """).data()

print("Observed relationship patterns:")
for p in patterns:
    print(p)


In [ ]:

import re
import torch

STOPWORDS = {
    "what", "which", "how", "many", "the", "a", "an", "is", "are",
    "in", "of", "to", "for", "all", "do", "does", "there", "have",
    "has", "with", "from", "stored", "database", "graph"
}

def question_words(text):
    return {
        word for word in re.findall(r"[a-z0-9_]+", text.lower())
        if len(word) > 1 and word not in STOPWORDS
    }

def retrieve_examples(question, candidates, k=3):
    qwords = question_words(question)
    scored = []

    for example in candidates:
        words = question_words(example["question"])
        union = qwords | words
        score = len(qwords & words) / max(len(union), 1)
        scored.append((score, example))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [example for score, example in scored[:k] if score > 0]


def generate_cypher_fewshot(question, candidates, k=3, max_new_tokens=160):
    selected = retrieve_examples(question, candidates, k=k)

    messages = [{
        "role": "system",
        "content": (
            "Generate exactly one read-only Cypher query for the provided graph schema. "
            "Use only the supplied labels, properties, and relationship directions. "
            "Do not invent schema elements. Return only the Cypher query.\n\n"
            "Schema:\n" + SCHEMA_TEXT
        ),
    }]

    for example in selected:
        messages.append({
            "role": "user",
            "content": "Translate this question into Cypher:\n" + example["question"],
        })
        messages.append({
            "role": "assistant",
            "content": example["cypher"],
        })

    messages.append({
        "role": "user",
        "content": "Translate this question into Cypher:\n" + question,
    })

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda:0")

    previous_cache = model.config.use_cache
    model.config.use_cache = True

    try:
        model.eval()
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    finally:
        model.config.use_cache = previous_cache

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return clean_cypher_output(answer), selected


# Test one question with retrieval-based prompting.
test_question = unseen_tests[0]["question"]

fewshot_query, retrieved = generate_cypher_fewshot(
    test_question,
    train_examples,
    k=3,
)

print("Question:", test_question)
print("\nRetrieved examples:")
for example in retrieved:
    print("-", example["question"])

print("\nFew-shot generated query:")
print(fewshot_query)

print("\nSafety check:")
print(safety_check(fewshot_query))

print("\nNeo4j result:")
print(execute_readonly(fewshot_query))


In [ ]:

assert "evaluation_results" in globals(), (
    "Run the unseen evaluation cell first."
)

for item in evaluation_results:
    print("\n" + "=" * 80)
    print("ID:", item.get("id"))
    print("Question:", item.get("question"))
    print("Status:", item.get("status"))

    print("\nExpected Cypher:")
    print(item.get("reference_cypher", item.get("expected_cypher", "Not recorded")))

    print("\nGenerated Cypher:")
    print(item.get("generated_cypher", "No query generated"))

    print("\nExpected results:")
    print(item.get("expected_rows", "Not recorded"))

    print("\nActual results:")
    print(item.get("actual_rows", "Not recorded"))

    print("\nFailure reason:")
    print(item.get("failure_reason",
                    item.get("generation_error",
                             item.get("reference_error", "None recorded"))))


In [ ]:

with driver.session(database=NEO4J_DATABASE) as session:
    patterns = session.run("""
        MATCH (a)-[r]->(b)
        RETURN DISTINCT
            labels(a) AS source_labels,
            type(r) AS relationship_type,
            labels(b) AS target_labels
        ORDER BY relationship_type
    """).data()

print("Observed relationship patterns:")
for p in patterns:
    print(p)


In [ ]:

assert "evaluation_results" in globals(), (
    "Run the unseen evaluation cell first."
)

for item in evaluation_results:
    print("\n" + "=" * 80)
    print("ID:", item.get("id"))
    print("Question:", item.get("question"))
    print("Status:", item.get("status"))

    print("\nExpected Cypher:")
    print(item.get("reference_cypher", item.get("expected_cypher", "Not recorded")))

    print("\nGenerated Cypher:")
    print(item.get("generated_cypher", "No query generated"))

    print("\nExpected results:")
    print(item.get("expected_rows", "Not recorded"))

    print("\nActual results:")
    print(item.get("actual_rows", "Not recorded"))

    print("\nFailure reason:")
    print(item.get("failure_reason",
                    item.get("generation_error",
                             item.get("reference_error", "None recorded"))))


In [ ]:

from neo4j import READ_ACCESS
import re

def execute_readonly(query, timeout_seconds=5):
    passed, reason = safety_check(query)

    if not passed:
        return {
            "success": False,
            "reason": reason,
            "rows": None,
        }

    query = query.strip()

    def read_work(tx):
        # EXPLAIN validates the query plan without running the data query.
        tx.run("EXPLAIN " + query).consume()

        result = tx.run(query)
        rows = []

        for record in result:
            rows.append(record.data())

            if len(rows) > 100:
                raise ValueError(
                    "Result exceeded the 100-row evaluation limit."
                )

        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            rows = session.execute_read(
                read_work,
                timeout=timeout_seconds,
            )

        return {
            "success": True,
            "reason": "Query executed successfully",
            "rows": rows,
        }

    except Exception as exc:
        return {
            "success": False,
            "reason": str(exc)[:500],
            "rows": None,
        }

print("Corrected read-only evaluation function defined.")


In [ ]:

test_result = execute_readonly("""
    MATCH (m:Movie)
    RETURN count(m) AS answer
""")

print(test_result)

assert test_result["success"], (
    "Reference query failed. Check the Neo4j connection, "
    "database name, driver version, and schema."
)


In [ ]:

from neo4j import READ_ACCESS

def execute_readonly(query, timeout_seconds=5):
    passed, reason = safety_check(query)

    if not passed:
        return {
            "success": False,
            "reason": reason,
            "rows": None,
        }

    query = query.strip()

    def read_work(tx):
        tx.run("EXPLAIN " + query).consume()

        result = tx.run(query)
        rows = []

        for record in result:
            if len(rows) >= 100:
                raise ValueError("Result exceeds the 100-row limit.")
            rows.append(record.data())

        return rows

    try:
        with driver.session(
            database=NEO4J_DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            # Do NOT pass timeout= to execute_read().
            rows = session.execute_read(read_work)

        return {
            "success": True,
            "reason": "Query executed successfully",
            "rows": rows,
        }

    except Exception as exc:
        return {
            "success": False,
            "reason": str(exc)[:500],
            "rows": None,
        }

print("execute_readonly() replaced without the unsupported timeout argument.")


In [ ]:

import inspect

print("Current function signature:")
print(inspect.signature(execute_readonly))

test_result = execute_readonly("""
    MATCH (m:Movie)
    RETURN count(m) AS answer
""")

print("\nTest result:")
print(test_result)


In [ ]:

evaluation_results = []

for test in unseen_tests:
    print("\n" + "=" * 65)
    print(test["id"], "-", test["question"])

    # Execute the independently prepared reference query first.
    reference = execute_readonly(test["reference_cypher"])

    if not reference["success"]:
        result = {
            **test,
            "status": "REFERENCE_QUERY_FAILED",
            "reference_error": reference["reason"],
        }
        evaluation_results.append(result)
        print("Reference query failed:", reference["reason"])
        continue

    try:
        generated = clean_cypher_output(
            generate_cypher(test["question"])
        )
    except Exception as exc:
        evaluation_results.append({
            **test,
            "status": "GENERATION_FAILED",
            "generation_error": str(exc)[:500],
            "expected_rows": reference["rows"],
        })
        print("Generation failed:", exc)
        continue

    model_result = execute_readonly(generated)

    if not model_result["success"]:
        evaluation_results.append({
            **test,
            "generated_cypher": generated,
            "status": "GENERATED_QUERY_FAILED",
            "failure_reason": model_result["reason"],
            "expected_rows": reference["rows"],
        })
        print("Generated query failed:", model_result["reason"])
        print("Generated Cypher:", generated)
        continue

    expected = canonical_values(reference["rows"])
    actual = canonical_values(model_result["rows"])
    correct = expected == actual

    evaluation_results.append({
        **test,
        "generated_cypher": generated,
        "expected_rows": reference["rows"],
        "actual_rows": model_result["rows"],
        "status": "PASS" if correct else "WRONG_ANSWER",
    })

    print("Expected:", reference["rows"])
    print("Actual:", model_result["rows"])
    print("Generated Cypher:", generated)
    print("Status:", "PASS" if correct else "WRONG_ANSWER")

print("\nEvaluation completed.")
print("Tests:", len(evaluation_results))
print("Correct:", sum(r.get("status") == "PASS" for r in evaluation_results))
print("Reference failures:", sum(
    r.get("status") == "REFERENCE_QUERY_FAILED"
    for r in evaluation_results
))


In [ ]:
%pip install -q gradio neo4j peft accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "google/gemma-3-4b-it"

# Change this to your actual saved adapter directory
ADAPTER_PATH = "/content/drive/MyDrive/content/querycraft_dataset/querycraft_gemma3_lora_adapter/adapter_config.json"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

model.eval()

print("QueryCraft model and LoRA adapter loaded.")

In [ ]:

from pathlib import Path

ADAPTER_PATH = Path("/content/querycraft_lora_adapter")

print("Adapter directory exists:", ADAPTER_PATH.exists())

if ADAPTER_PATH.exists():
    print("\nFiles inside adapter directory:")
    for item in sorted(ADAPTER_PATH.iterdir()):
        print(item.name)
else:
    print("\nAdapter directory not found.")
    print("\nSearching /content for adapter configuration files:")

    for item in Path("/content").rglob("adapter_config.json"):
        print("Found:", item)


In [ ]:
from pathlib import Path

ADAPTER_PATH = Path(
    "/content/drive/MyDrive/QueryCraft_Gemma3_4B/final_adapter"
)

print("Adapter directory exists:", ADAPTER_PATH.exists())
print("Adapter configuration exists:",
      (ADAPTER_PATH / "adapter_config.json").is_file())
print("Adapter weights exist:",
      (ADAPTER_PATH / "adapter_model.safetensors").is_file())

if ADAPTER_PATH.exists():
    print("\nFiles:")
    for item in sorted(ADAPTER_PATH.iterdir()):
        print(item.name)

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from pathlib import Path

BASE_MODEL = "google/gemma-3-4b-it"
ADAPTER_PATH = "/content/drive/MyDrive/QueryCraft_Gemma3_4B/final_adapter"

adapter_dir = Path(ADAPTER_PATH)

assert (adapter_dir / "adapter_config.json").is_file(), (
    f"adapter_config.json missing from {adapter_dir}"
)

assert (
    (adapter_dir / "adapter_model.safetensors").is_file()
    or (adapter_dir / "adapter_model.bin").is_file()
), "LoRA adapter weights not found."

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
)

model.eval()

print("QueryCraft Gemma 3 4B LoRA adapter loaded successfully.")


In [ ]:

prompt = """Convert this question into a Cypher query.
Question: How many movies are in the database?
Cypher:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.get_input_embeddings().weight.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Question: How many movies are in the database?")
print("\nGenerated Cypher:")
print(response)


In [ ]:

import torch

SCHEMA = """
Nodes:
- Movie(title, released)
- Person
- Genre

Relationships:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)
"""

def generate_cypher(question):
    messages = [
        {
            "role": "user",
            "content": f"""
You are QueryCraft, a natural-language-to-Neo4j-Cypher translator.

GRAPH SCHEMA:
{SCHEMA}

VALID EXAMPLE:
Question: What is the release year of The Matrix?
Cypher:
MATCH (m:Movie {{title: 'The Matrix'}})
RETURN m.released AS release_year

RULES:
1. Return exactly one read-only Neo4j Cypher query.
2. Use only labels, relationships, and properties declared in the schema.
3. Do not use SQL syntax, including JOIN, ON, or BETWEEN.
4. Do not invent properties such as Person.name or Genre.name.
5. For actor or director questions, return the Person node
   unless a Person property has been verified in the schema.
6. For genre questions, use:
   (Movie)-[:IN_GENRE]->(Genre)
7. For movie release-year ranges, use comparison conditions.
8. Ensure parentheses, brackets, and braces are correctly balanced.
9. Do not use CREATE, MERGE, DELETE, DETACH, SET, REMOVE,
   DROP, CALL, LOAD CSV, or multiple statements.
10. Return no explanations or Markdown fences.

EXAMPLES:

Question: Which people directed The Matrix?
Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {{title: 'The Matrix'}})
RETURN p

Question: Which movies belong to a genre?
Cypher:
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
RETURN m.title AS movie_title

Question: Find movies released between 2000 and 2010.
Cypher:
MATCH (m:Movie)
WHERE m.released >= 2000 AND m.released <= 2010
RETURN m.title AS movie_title

QUESTION:
{question}

Cypher:
"""
        }
    ]

    prompt = test_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    end_turn_id = test_tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )

    eos_ids = [test_tokenizer.eos_token_id]

    if (
        end_turn_id is not None
        and end_turn_id != test_tokenizer.unk_token_id
        and end_turn_id not in eos_ids
    ):
        eos_ids.append(end_turn_id)

    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=test_tokenizer.pad_token_id
        )

    new_tokens = output[
        0, inputs["input_ids"].shape[1]:
    ]

    return test_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

print("Updated generate_cypher() function is ready.")


In [ ]:
# QueryCraft: Check Gemma 3 4B IT chat template

print("Tokenizer exists:", "tokenizer" in globals())
print("Processor exists:", "processor" in globals())

if "tokenizer" in globals():
    print("\nTokenizer class:", type(tokenizer).__name__)
    print("Tokenizer has chat_template:",
          getattr(tokenizer, "chat_template", None) is not None)

if "processor" in globals():
    print("\nProcessor class:", type(processor).__name__)
    print("Processor has chat_template:",
          getattr(processor, "chat_template", None) is not None)

In [ ]:
# QueryCraft: Verify Gemma 3 4B IT chat template

print("Tokenizer class:", type(tokenizer).__name__)
print("Chat template available:", tokenizer.chat_template is not None)

messages = [
    {
        "role": "user",
        "content": "Generate Cypher to count all Movie nodes."
    }
]

try:
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("\nSUCCESS: Chat template works.")
    print("\nFormatted prompt:")
    print(repr(formatted_prompt))

except Exception as e:
    print("\nChat template test failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Generate Cypher to count all Movie nodes."
    }
]

try:
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("SUCCESS: Tokenizer chat template works.")
    print("\nFormatted prompt:\n")
    print(formatted_prompt)

except Exception as e:
    print("Tokenizer chat template test failed:")
    print(type(e).__name__, str(e))

In [ ]:
print("Model exists:", "model" in globals())

if "model" in globals():
    print("Model class:", type(model).__name__)
    print("Model is in evaluation mode:", not model.training)

In [ ]:

import torch

def generate_cypher(question):
    messages = [
        {
            "role": "user",
            "content": f"""
You are QueryCraft, a natural-language-to-Cypher assistant.

Use this Neo4j graph schema:
- Nodes: Movie, Person, Genre
- Relationships: (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)

Generate one read-only Cypher query for the question below.
Return only the Cypher query, without Markdown.
Do not use CREATE, DELETE, SET, MERGE, DROP, or administrative procedures.

Question: {question}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


test_question = "How many movies are in the database?"
print(generate_cypher(test_question))


In [ ]:

import torch

print("Model exists:", "model" in globals())
print("Tokenizer exists:", "tokenizer" in globals())

messages = [{
    "role": "user",
    "content": "Return only this Cypher query: MATCH (m:Movie) RETURN count(m) AS answer"
}]

try:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    print("Prompt prepared successfully.")
    print("Input token count:", inputs["input_ids"].shape[1])

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_length:]

    print("New tokens generated:", new_tokens.numel())
    print("Raw output:", repr(
        tokenizer.decode(new_tokens, skip_special_tokens=False)
    ))
    print("Clean output:", repr(
        tokenizer.decode(new_tokens, skip_special_tokens=True)
    ))

except Exception as e:
    print("ERROR:", type(e).__name__, str(e))


In [ ]:

import torch

def generate_cypher(question):
    messages = [
        {
            "role": "user",
            "content": f"""
You are QueryCraft, a natural-language-to-Neo4j-Cypher translator.

GRAPH SCHEMA:
Nodes:
- Movie(title, released)
- Person
- Genre

Relationships:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)

RULES:
1. Return exactly one read-only Neo4j Cypher query.
2. Use only the labels, relationships, and properties declared above.
3. Do not use SQL syntax.
4. Do not invent properties such as Person.name or Genre.name.
5. Use the correct relationship direction from the schema.
6. Ensure parentheses, brackets, and braces are balanced.
7. Do not use CREATE, MERGE, DELETE, DETACH, SET, REMOVE,
   DROP, CALL, LOAD CSV, or multiple statements.
8. Return only Cypher, without explanations or Markdown fences.

EXAMPLE 1 — Release year:
Question: What is the release year of The Matrix?
Cypher:
MATCH (m:Movie {{title: 'The Matrix'}})
RETURN m.released AS release_year

EXAMPLE 2 — Director:
Question: Which people directed The Matrix?
Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {{title: 'The Matrix'}})
RETURN p

EXAMPLE 3 — Genre:
Question: Which movies belong to a genre?
Cypher:
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
RETURN m.title AS movie_title

EXAMPLE 4 — Movies with actors:
Question: Which movies have at least one actor?
Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
RETURN DISTINCT m.title AS movie_title

EXAMPLE 5 — Release-year range:
Question: Find movies released between 2000 and 2010.
Cypher:
MATCH (m:Movie)
WHERE m.released >= 2000 AND m.released <= 2010
RETURN m.title AS movie_title

Now translate this question using the schema and rules above.

Question: {question}

Cypher:
"""
        }
    ]

    prompt = test_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    end_turn_id = test_tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )

    eos_ids = [test_tokenizer.eos_token_id]

    if (
        end_turn_id is not None
        and end_turn_id != test_tokenizer.unk_token_id
        and end_turn_id not in eos_ids
    ):
        eos_ids.append(end_turn_id)

    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=test_tokenizer.pad_token_id
        )

    new_tokens = output[
        0, inputs["input_ids"].shape[1]:
    ]

    return test_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

print("QueryCraft generate_cypher() updated successfully.")


In [ ]:
question = "Which movies have at least one actor?"

generated_query = generate_cypher(question)

print("Generated Cypher:")
print(generated_query)

In [ ]:

# QueryCraft: Diagnose PAD-only generation

print("=== TOKENIZER ===")
print("Tokenizer class:", type(tokenizer).__name__)
print("PAD token:", repr(tokenizer.pad_token))
print("PAD token ID:", tokenizer.pad_token_id)
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("BOS token ID:", tokenizer.bos_token_id)
print("Tokenizer vocabulary length:", len(tokenizer))

print("\n=== MODEL ===")
print("Model class:", type(model).__name__)
print("Model vocabulary size:", model.config.vocab_size)
print("Input embedding shape:", tuple(
    model.get_input_embeddings().weight.shape
))

print("\n=== GENERATION CONFIG ===")
print("Model generation PAD ID:",
      model.generation_config.pad_token_id)
print("Model generation EOS ID:",
      model.generation_config.eos_token_id)

print("\n=== PEFT ADAPTER ===")
if hasattr(model, "peft_config"):
    print("Available adapters:", list(model.peft_config.keys()))
    print("Active adapter:", getattr(model, "active_adapter", "Unknown"))
else:
    print("No PEFT adapter configuration found.")


In [ ]:

# QueryCraft: Safe Gemma 3 / PEFT diagnostics

print("=== MODEL CONFIGURATION ===")

config = model.config
print("Config class:", type(config).__name__)
print("Model type:", getattr(config, "model_type", "Unknown"))

text_config = getattr(config, "text_config", None)

if text_config is not None:
    print("Text config class:", type(text_config).__name__)
    print("Text vocabulary size:",
          getattr(text_config, "vocab_size", "Not exposed"))
else:
    print("Text vocabulary size:",
          getattr(config, "vocab_size", "Not exposed"))

try:
    embedding = model.get_input_embeddings()
    print("Input embedding shape:", tuple(embedding.weight.shape))
except Exception as e:
    print("Embedding inspection error:", type(e).__name__, str(e))

print("\n=== TOKENIZER ===")
print("Tokenizer vocabulary length:", len(tokenizer))
print("PAD ID:", tokenizer.pad_token_id)
print("EOS ID:", tokenizer.eos_token_id)
print("BOS ID:", tokenizer.bos_token_id)

print("\n=== GENERATION CONFIG ===")
print("PAD ID:", model.generation_config.pad_token_id)
print("EOS ID:", model.generation_config.eos_token_id)

print("\n=== PEFT ADAPTER ===")
print("Available adapters:", list(model.peft_config.keys()))
print("Active adapter:", getattr(model, "active_adapter", "Unknown"))


In [ ]:

# QueryCraft: Inspect model output scores

try:
    with torch.inference_mode():
        result = model(**inputs)

    logits = result.logits[:, -1, :].float()

    print("Logits shape:", tuple(logits.shape))
    print("All logits finite:",
          bool(torch.isfinite(logits).all()))
    print("Contains NaN:",
          bool(torch.isnan(logits).any()))
    print("Contains Inf:",
          bool(torch.isinf(logits).any()))

    top_count = min(10, logits.shape[-1])
    scores, token_ids = torch.topk(logits[0], k=top_count)

    print("\nTop next-token predictions:")
    for score, token_id in zip(scores.tolist(), token_ids.tolist()):
        print(
            "ID:", token_id,
            "| Token:", repr(tokenizer.convert_ids_to_tokens(token_id)),
            "| Score:", score
        )

except Exception as e:
    print("Inspection error:", type(e).__name__, str(e))


In [ ]:

# QueryCraft: Isolate base-model vs LoRA NaN issue

import torch

print("=== INPUT CHECK ===")
print("Input device:", inputs["input_ids"].device)
print("Model device:", model.device)
print("Model dtype:", next(model.parameters()).dtype)
print("Input token IDs min:", inputs["input_ids"].min().item())
print("Input token IDs max:", inputs["input_ids"].max().item())

def check_logits(label, current_model):
    try:
        with torch.inference_mode():
            output = current_model(**inputs)

        logits = output.logits[:, -1, :].float()

        print(f"\n{label}")
        print("Logits shape:", tuple(logits.shape))
        print("All finite:", torch.isfinite(logits).all().item())
        print("NaN present:", torch.isnan(logits).any().item())
        print("Inf present:", torch.isinf(logits).any().item())

        if torch.isfinite(logits).any():
            safe_logits = logits.masked_fill(
                ~torch.isfinite(logits),
                float("-inf")
            )
            scores, ids = torch.topk(
                safe_logits[0], k=5
            )
            print("Top tokens:")
            for score, token_id in zip(
                scores.tolist(), ids.tolist()
            ):
                print(
                    token_id,
                    repr(tokenizer.convert_ids_to_tokens(token_id)),
                    score
                )

    except Exception as e:
        print(label, "ERROR:", type(e).__name__, str(e))


# Test the currently active adapter
check_logits("ADAPTER ENABLED", model)

# Temporarily disable PEFT adapter; no files are changed
try:
    with model.disable_adapter():
        check_logits("ADAPTER DISABLED (BASE MODEL)", model)
except Exception as e:
    print("\nCould not disable adapter:", type(e).__name__, str(e))


In [ ]:

# QueryCraft: Inspect base-model parameter integrity

base_model = model.get_base_model()

print("Base model class:", type(base_model).__name__)
print("Model architectures:",
      getattr(base_model.config, "architectures", None))
print("Model dtype:", next(base_model.parameters()).dtype)

print("\n=== SELECTED PARAMETER CHECKS ===")

checked = 0
nonfinite = []

for name, param in base_model.named_parameters():
    if any(key in name.lower() for key in [
        "embed_tokens", "norm", "lm_head", "q_proj", "v_proj"
    ]):
        tensor = param.detach()
        finite = torch.isfinite(tensor).all().item()
        checked += 1

        print(
            name,
            "| shape:", tuple(tensor.shape),
            "| dtype:", tensor.dtype,
            "| finite:", finite
        )

        if not finite:
            nonfinite.append(name)

        if checked >= 12:
            break

print("\nParameters checked:", checked)
print("Non-finite parameter names:", nonfinite)


In [ ]:

import os
import json
from pathlib import Path

ADAPTER_DIR = (
    "/content/drive/MyDrive/content/querycraft_dataset/"
    "querycraft_gemma3_lora_adapter"
)

adapter_path = Path(ADAPTER_DIR)

print("=== ADAPTER DIRECTORY ===")
print("Directory exists:", adapter_path.is_dir())

if adapter_path.is_dir():
    for path in sorted(adapter_path.iterdir()):
        print(path.name, "-", "directory" if path.is_dir() else
              f"{path.stat().st_size:,} bytes")

    config_path = adapter_path / "adapter_config.json"

    print("\n=== ADAPTER CONFIGURATION ===")
    if config_path.exists():
        with open(config_path, "r") as f:
            adapter_config = json.load(f)

        for key in [
            "base_model_name_or_path",
            "peft_type",
            "task_type",
            "r",
            "lora_alpha",
            "target_modules",
        ]:
            print(f"{key}: {adapter_config.get(key, 'Not specified')}")
    else:
        print("adapter_config.json is missing.")

else:
    print("Check the Google Drive mount and adapter path.")


In [ ]:

from pathlib import Path

adapter_path = Path(ADAPTER_DIR)

weight_files = (
    list(adapter_path.glob("adapter_model.safetensors"))
    + list(adapter_path.glob("adapter_model.bin"))
)

print("=== ADAPTER WEIGHT FILES ===")

if not weight_files:
    print("No standard PEFT adapter weight file found.")
else:
    for weight_file in weight_files:
        print("File:", weight_file.name)

        if weight_file.suffix == ".safetensors":
            from safetensors import safe_open

            with safe_open(
                str(weight_file),
                framework="pt",
                device="cpu"
            ) as f:
                keys = list(f.keys())

        else:
            import torch
            state = torch.load(
                weight_file,
                map_location="cpu",
                weights_only=True
            )
            keys = list(state.keys())

        print("Number of saved tensors:", len(keys))
        print("First 15 tensor names:")
        for key in keys[:15]:
            print(" ", key)

        lora_keys = [
            key for key in keys
            if "lora_A" in key or "lora_B" in key
        ]
        print("LoRA tensor count:", len(lora_keys))


In [ ]:

import json
from pathlib import Path
from safetensors import safe_open

adapter_path = Path(ADAPTER_DIR)

print("=== FULL ADAPTER CONFIG ===")
with open(adapter_path / "adapter_config.json") as f:
    cfg = json.load(f)

print(json.dumps(cfg, indent=2))

print("\n=== REPRESENTATIVE LORA WEIGHT SHAPES ===")
with safe_open(
    str(adapter_path / "adapter_model.safetensors"),
    framework="pt",
    device="cpu"
) as f:
    keys = list(f.keys())

    for key in keys[:8]:
        print(key, "->", tuple(f.get_tensor(key).shape))

print("\nTotal saved tensors:", len(keys))

print("\n=== SAVED TOKENIZER CONFIG ===")
with open(adapter_path / "tokenizer_config.json") as f:
    tok_cfg = json.load(f)

for key in [
    "tokenizer_class",
    "model_max_length",
    "bos_token",
    "eos_token",
    "pad_token",
]:
    print(f"{key}: {tok_cfg.get(key, 'Not specified')}")

print("\n=== ADAPTER README: FIRST 25 LINES ===")
readme = adapter_path / "README.md"
if readme.exists():
    print("\n".join(readme.read_text().splitlines()[:25]))


In [ ]:

import json
import shutil
from pathlib import Path
from safetensors.torch import load_file, save_file

SOURCE_DIR = Path(ADAPTER_DIR)
REPAIRED_DIR = SOURCE_DIR.parent / "querycraft_gemma3_lora_adapter_repaired"

assert (SOURCE_DIR / "adapter_config.json").exists()
assert (SOURCE_DIR / "adapter_model.safetensors").exists()
assert not REPAIRED_DIR.exists(), (
    f"{REPAIRED_DIR} already exists. Choose a new output directory."
)

# Copy the original adapter files to a new directory.
shutil.copytree(SOURCE_DIR, REPAIRED_DIR)

# Read the saved weights.
weights = load_file(
    str(SOURCE_DIR / "adapter_model.safetensors"),
    device="cpu"
)

# Remove exactly one duplicated prefix.
old_prefix = "base_model.model.base_model.model."
new_prefix = "base_model.model."

fixed_weights = {}
for key, tensor in weights.items():
    if key.startswith(old_prefix):
        new_key = new_prefix + key[len(old_prefix):]
    else:
        new_key = key

    if new_key in fixed_weights:
        raise ValueError(f"Duplicate key after repair: {new_key}")

    fixed_weights[new_key] = tensor

# Verify the repair actually changed the expected keys.
changed = sum(
    old != new
    for old, new in zip(weights.keys(), fixed_weights.keys())
)

print("Original tensor count:", len(weights))
print("Repaired tensor count:", len(fixed_weights))
print("Keys renamed:", changed)
print("Example repaired key:", next(iter(fixed_weights)))

assert len(weights) == len(fixed_weights)
assert all(
    not key.startswith(old_prefix)
    for key in fixed_weights
)

save_file(
    fixed_weights,
    str(REPAIRED_DIR / "adapter_model.safetensors")
)

# Set the documented base model explicitly.
config_path = REPAIRED_DIR / "adapter_config.json"
with open(config_path) as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = "google/gemma-3-1b-it"

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("\nRepaired adapter saved to:", REPAIRED_DIR)
print("Original adapter preserved:", SOURCE_DIR.exists())


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "google/gemma-3-1b-it"

assert torch.cuda.is_available(), "Enable a GPU first."

test_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

base_model.eval()

test_messages = [{
    "role": "user",
    "content": "Return only: MATCH (m:Movie) RETURN count(m) AS answer"
}]

test_prompt = test_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

test_inputs = test_tokenizer(
    test_prompt,
    return_tensors="pt"
).to(base_model.device)

with torch.inference_mode():
    test_output = base_model(**test_inputs)

test_logits = test_output.logits[:, -1, :].float()

print("Base model:", BASE_MODEL)
print("Model class:", type(base_model).__name__)
print("Logits finite:", torch.isfinite(test_logits).all().item())
print("NaN present:", torch.isnan(test_logits).any().item())
print("Inf present:", torch.isinf(test_logits).any().item())

assert torch.isfinite(test_logits).all(), (
    "Base model produces non-finite logits. Stop here; "
    "do not load the adapter until the base-model issue is resolved."
)

print("Base-model numerical check passed.")


In [ ]:

from peft import PeftModel
import torch

print("Repaired adapter directory:", REPAIRED_DIR)
print("Directory exists:", REPAIRED_DIR.exists())

try:
    model = PeftModel.from_pretrained(
        base_model,
        str(REPAIRED_DIR),
        is_trainable=False,
    )

    model.eval()
    print("Adapter loaded successfully.")

    with torch.inference_mode():
        adapted_output = model(**test_inputs)

    adapted_logits = adapted_output.logits[:, -1, :].float()

    print("Adapted logits finite:",
          torch.isfinite(adapted_logits).all().item())
    print("NaN present:",
          torch.isnan(adapted_logits).any().item())
    print("Inf present:",
          torch.isinf(adapted_logits).any().item())

    if torch.isfinite(adapted_logits).all():
        with torch.inference_mode():
            generated = model.generate(
                **test_inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=test_tokenizer.pad_token_id,
                eos_token_id=test_tokenizer.eos_token_id,
            )

        new_tokens = generated[0, test_inputs["input_ids"].shape[1]:]

        print("\nGenerated output:")
        print(repr(test_tokenizer.decode(
            new_tokens, skip_special_tokens=True
        )))

except Exception as e:
    print("ADAPTER TEST FAILED")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))


In [ ]:

import torch

SCHEMA = """
Node labels: Movie, Person, Genre
Relationships:
(Person)-[:ACTED_IN]->(Movie)
(Person)-[:DIRECTED]->(Movie)
(Movie)-[:IN_GENRE]->(Genre)
Movie properties include title and released.
"""

question = "What is the release year of the movie named The Matrix?"

messages = [
    {
        "role": "user",
        "content": f"""You translate questions into Neo4j Cypher.

Graph schema:
{SCHEMA}

Rules:
- Return exactly one read-only Cypher query.
- Use only labels, relationships and properties in the schema.
- Do not include explanations or Markdown fences.
- Begin with MATCH or another appropriate Cypher clause.
- Return the movie's release year using the alias release_year.

Question: {question}"""
    }
]

prompt = test_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = test_tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

end_turn_id = test_tokenizer.convert_tokens_to_ids("<end_of_turn>")
eos_ids = [test_tokenizer.eos_token_id]

if (
    end_turn_id is not None
    and end_turn_id != test_tokenizer.unk_token_id
    and end_turn_id not in eos_ids
):
    eos_ids.append(end_turn_id)

with torch.inference_mode():
    output = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=False,
        eos_token_id=eos_ids,
        pad_token_id=test_tokenizer.pad_token_id,
    )

new_tokens = output[0, model_inputs["input_ids"].shape[1]:]
generated_text = test_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Question:", question)
print("\nGenerated text:")
print(repr(generated_text))
print("\nReadable output:")
print(generated_text)


In [ ]:

import torch

SCHEMA = """
Nodes:
- Movie(title, released)
- Person
- Genre

Relationships:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)
"""

messages = [{
    "role": "user",
    "content": f"""You are QueryCraft, a Neo4j Cypher generator.

GRAPH SCHEMA:
{SCHEMA}

VALID EXAMPLE:
Question: What is the release year of The Matrix?
Cypher:
MATCH (m:Movie {{title: 'The Matrix'}})
RETURN m.released AS release_year

STRICT RULES:
- Output only one Cypher query.
- Use Neo4j Cypher syntax, never SQL.
- Do not use JOIN, ON, or SQL-style table aliases.
- Use only the node labels, relationships, and properties listed above.
- Do not invent properties such as release_year; use released.
- Do not add an explanation or Markdown fences.

Question: {question}
Cypher:"""
}]

prompt = test_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = test_tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    output = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=False,
        eos_token_id=[
            test_tokenizer.eos_token_id,
            test_tokenizer.convert_tokens_to_ids("<end_of_turn>")
        ],
        pad_token_id=test_tokenizer.pad_token_id,
    )

new_tokens = output[0, model_inputs["input_ids"].shape[1]:]
generated_text = test_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Generated Cypher:\n")
print(generated_text)


In [ ]:
import re

def check_release_year_query(query):
    normalized = " ".join(query.strip().split())

    valid = bool(
        re.search(
            r"MATCH\s*\(\s*m\s*:\s*Movie\s*"
            r"\{\s*title\s*:\s*['\"]The Matrix['\"]\s*\}\s*\)"
            r".*RETURN\s+m\.released\s+AS\s+release_year\b",
            normalized,
            flags=re.IGNORECASE,
        )
    )

    print("Expected query format:", "PASS" if valid else "FAIL")
    return valid

check_release_year_query(generated_text)

In [ ]:
print("Generated query:")
print(repr(generated_text))

In [ ]:
import re

def check_release_year_query(query):
    normalized = " ".join(query.strip().split())

    print("Actual query:", normalized)

    has_match = bool(
        re.search(
            r"\bMATCH\s*\(\s*m\s*:\s*Movie\b",
            normalized,
            re.IGNORECASE,
        )
    )

    has_title = bool(
        re.search(
            r"title\s*:\s*['\"]The Matrix['\"]",
            normalized,
            re.IGNORECASE,
        )
    )

    has_return = bool(
        re.search(
            r"\bRETURN\s+m\.released\s+AS\s+release_year\b",
            normalized,
            re.IGNORECASE,
        )
    )

    print("Movie MATCH clause:", "PASS" if has_match else "FAIL")
    print("Movie title filter:", "PASS" if has_title else "FAIL")
    print("Release year alias:", "PASS" if has_return else "FAIL")

    passed = has_match and has_title and has_return
    print("Overall validation:", "PASS" if passed else "FAIL")

    return passed

check_release_year_query(generated_text)

In [ ]:

import torch

SCHEMA = """
Nodes:
- Movie(title, released)
- Person
- Genre

Relationships:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)
"""

question = "What is the release year of the movie named The Matrix?"

messages = [
    {
        "role": "user",
        "content": f"""
You are QueryCraft, a natural-language-to-Neo4j-Cypher translator.

Graph schema:
{SCHEMA}

Example:
Question: What is the release year of The Matrix?
Cypher:
MATCH (m:Movie {{title: 'The Matrix'}})
RETURN m.released AS release_year

Rules:
1. Output exactly one read-only Neo4j Cypher query.
2. Use only labels, relationships, and properties from the schema.
3. Do not use SQL syntax such as JOIN or ON.
4. Do not invent properties. The Movie property is released, not release_year.
5. Do not include explanations or Markdown fences.
6. For this question, the RETURN clause MUST be exactly:
   RETURN m.released AS release_year

Question: {question}

Cypher:
"""
    }
]

# Format the prompt using Gemma's chat template
prompt = test_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and send inputs to the model's device
model_inputs = test_tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Stop generation at the end of the assistant turn
end_turn_id = test_tokenizer.convert_tokens_to_ids("<end_of_turn>")

eos_ids = [test_tokenizer.eos_token_id]

if (
    end_turn_id is not None
    and end_turn_id != test_tokenizer.unk_token_id
    and end_turn_id not in eos_ids
):
    eos_ids.append(end_turn_id)

# Generate a fresh response
model.eval()

with torch.inference_mode():
    output = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
        eos_token_id=eos_ids,
        pad_token_id=test_tokenizer.pad_token_id,
    )

# Decode only newly generated tokens
new_tokens = output[
    0, model_inputs["input_ids"].shape[1]:
]

generated_text = test_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Generated Cypher:")
print(generated_text)


In [ ]:
check_release_year_query(generated_text)

In [ ]:

import re
import torch

# --------------------------------------------------
# QueryCraft: Model-only evaluation
# Prerequisites: model and test_tokenizer are loaded
# --------------------------------------------------

SCHEMA = """
Nodes:
- Movie(title, released)
- Person
- Genre

Relationships:
- (Person)-[:ACTED_IN]->(Movie)
- (Person)-[:DIRECTED]->(Movie)
- (Movie)-[:IN_GENRE]->(Genre)
"""

TESTS = [
    {
        "id": "Q01",
        "question": "What is the release year of The Matrix?",
        "expected": r"RETURN\s+m\.released\s+AS\s+release_year"
    },
    {
        "id": "Q02",
        "question": "Find all movies released in 1999.",
        "expected": r"m\.released\s*=\s*1999"
    },
    {
        "id": "Q03",
        "question": "List all movie titles.",
        "expected": r"RETURN\s+m\.title\b"
    },
    {
        "id": "Q04",
        "question": "Which people acted in The Matrix?",
        "expected": r"\[:ACTED_IN\]"
    },
    {
        "id": "Q05",
        "question": "Which people directed The Matrix?",
        "expected": r"\[:DIRECTED\]"
    },
    {
        "id": "Q06",
        "question": "Which movies belong to a genre?",
        "expected": r"\[:IN_GENRE\]"
    },
    {
        "id": "Q07",
        "question": "Find movies released after 2010.",
        "expected": r"m\.released\s*>\s*2010"
    },
    {
        "id": "Q08",
        "question": "How many movies are in the database?",
        "expected": r"count\s*\(\s*m\s*\)"
    },
    {
        "id": "Q09",
        "question": "Which movies have at least one actor?",
        "expected": r"\[:ACTED_IN\]"
    },
    {
        "id": "Q10",
        "question": "Find movies released between 2000 and 2010.",
        "expected": r"m\.released\s*>=?\s*2000"
    },
]

def generate_cypher(question):
    messages = [
        {
            "role": "user",
            "content": f"""
You translate natural-language questions into Neo4j Cypher.

Graph schema:
{SCHEMA}

Example:
Question: What is the release year of The Matrix?
Cypher:
MATCH (m:Movie {{title: 'The Matrix'}})
RETURN m.released AS release_year

Rules:
- Return exactly one read-only Cypher query.
- Use only schema labels, relationships, and properties.
- Use valid Neo4j Cypher syntax, not SQL.
- Do not invent properties.
- Use MATCH, OPTIONAL MATCH, or another appropriate Cypher clause.
- Do not use CREATE, MERGE, DELETE, DETACH, SET, REMOVE,
  DROP, CALL, LOAD CSV, or multiple statements.
- Do not include explanations or Markdown fences.

Question: {question}

Cypher:
"""
        }
    ]

    prompt = test_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    end_turn_id = test_tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )

    eos_ids = [test_tokenizer.eos_token_id]

    if (
        end_turn_id is not None
        and end_turn_id != test_tokenizer.unk_token_id
        and end_turn_id not in eos_ids
    ):
        eos_ids.append(end_turn_id)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=test_tokenizer.pad_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    return test_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


# Basic safety screening, not a full Cypher security parser.
WRITE_OR_ADMIN = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|"
    r"CALL|LOAD\s+CSV|FOREACH|GRANT|DENY|REVOKE)\b",
    re.IGNORECASE
)

def evaluate_query(query, expected_pattern):
    # Remove common Markdown fences, if generated.
    cleaned = re.sub(r"```(?:cypher)?", "", query,
                     flags=re.IGNORECASE).strip()
    cleaned = cleaned.replace("```", "").strip()

    # Reject multiple statements.
    statement = cleaned.rstrip(";")
    has_multiple_statements = ";" in statement

    read_only = (
        not WRITE_OR_ADMIN.search(statement)
        and not has_multiple_statements
    )

    schema_ok = (
        re.search(r":\s*Movie\b", statement, re.IGNORECASE)
        is not None
        or re.search(r":\s*Person\b", statement, re.IGNORECASE)
        is not None
        or re.search(r":\s*Genre\b", statement, re.IGNORECASE)
        is not None
        or re.search(r"\bMATCH\b", statement, re.IGNORECASE)
        is not None
    )

    expected_ok = bool(
        re.search(expected_pattern, statement, re.IGNORECASE)
    )

    starts_correctly = bool(
        re.match(
            r"^(MATCH|OPTIONAL\s+MATCH|WITH|RETURN)\b",
            statement,
            re.IGNORECASE
        )
    )

    passed = (
        read_only
        and schema_ok
        and expected_ok
        and starts_correctly
    )

    return {
        "query": statement,
        "read_only_screen": read_only,
        "expected_pattern": expected_ok,
        "starts_correctly": starts_correctly,
        "result": "PASS" if passed else "REVIEW"
    }


# Run tests
model.eval()
results = []

for test in TESTS:
    print(f"\n{'=' * 60}")
    print(f"{test['id']}: {test['question']}")

    try:
        query = generate_cypher(test["question"])
        checks = evaluate_query(query, test["expected"])

        print("Generated query:")
        print(query or "[EMPTY OUTPUT]")
        print("Read-only screening:", checks["read_only_screen"])
        print("Expected pattern:", checks["expected_pattern"])
        print("Starts correctly:", checks["starts_correctly"])
        print("Result:", checks["result"])

        results.append({
            "ID": test["id"],
            "Question": test["question"],
            "Query": checks["query"],
            "Read-only screen": checks["read_only_screen"],
            "Expected pattern": checks["expected_pattern"],
            "Result": checks["result"]
        })

    except Exception as e:
        print("ERROR:", repr(e))
        results.append({
            "ID": test["id"],
            "Question": test["question"],
            "Query": "",
            "Read-only screen": False,
            "Expected pattern": False,
            "Result": "ERROR"
        })

print("\n\n========== EVALUATION SUMMARY ==========")
passed = sum(r["Result"] == "PASS" for r in results)
print(f"Passed: {passed}/{len(TESTS)}")
print(f"Needs review: {len(TESTS) - passed}/{len(TESTS)}")

try:
    import pandas as pd
    display(pd.DataFrame(results))
except ImportError:
    for item in results:
        print(item)


In [ ]:

RETEST_QUESTIONS = [
    ("Q05", "Which people directed The Matrix?"),
    ("Q06", "Which movies belong to a genre?"),
    ("Q10", "Find movies released between 2000 and 2010.")
]

for test_id, question in RETEST_QUESTIONS:
    print("\n" + "=" * 60)
    print(f"{test_id}: {question}")

    generated_query = generate_cypher(question)

    print("\nGenerated Cypher:")
    print(generated_query)


In [ ]:

import re
import pandas as pd

# QueryCraft full regression evaluation
# Prerequisites:
# - model is loaded
# - test_tokenizer is loaded
# - updated generate_cypher() is defined

TESTS = [
    {
        "id": "Q01",
        "question": "What is the release year of The Matrix?",
        "expected": r"\bRETURN\s+m\.released\s+AS\s+release_year\b",
        "required": ["MATCH", "Movie"],
        "forbidden": []
    },
    {
        "id": "Q02",
        "question": "Find all movies released in 1999.",
        "expected": r"\bm\.released\s*=\s*1999\b",
        "required": ["MATCH", "Movie", "WHERE"],
        "forbidden": []
    },
    {
        "id": "Q03",
        "question": "List all movie titles.",
        "expected": r"\bRETURN\s+m\.title\b",
        "required": ["MATCH", "Movie"],
        "forbidden": []
    },
    {
        "id": "Q04",
        "question": "Which people acted in The Matrix?",
        "expected": r"\[:ACTED_IN\]",
        "required": ["MATCH", "Person", "Movie"],
        "forbidden": []
    },
    {
        "id": "Q05",
        "question": "Which people directed The Matrix?",
        "expected": r"\[:DIRECTED\]",
        "required": ["MATCH", "Person", "Movie"],
        "forbidden": []
    },
    {
        "id": "Q06",
        "question": "Which movies belong to a genre?",
        "expected": r"\[:IN_GENRE\]",
        "required": ["MATCH", "Movie", "Genre"],
        "forbidden": [r"\[:DIRECTED\]", r"\[:ACTED_IN\]"]
    },
    {
        "id": "Q07",
        "question": "Find movies released after 2010.",
        "expected": r"\bm\.released\s*>\s*2010\b",
        "required": ["MATCH", "Movie", "WHERE"],
        "forbidden": []
    },

{
        "id": "Q08",
        "question": "How many movies are in the database?",
        "expected": r"\bcount\s*\(\s*(?:m|\*)\s*\)",
        "required": ["MATCH", "Movie", "RETURN"],
        "forbidden": []
},
    {
        "id": "Q09",
        "question": "Which movies have at least one actor?",
        "expected": r"\[:ACTED_IN\]",
        "required": ["MATCH", "Person", "Movie"],
        "forbidden": []
    },
    {
        "id": "Q10",
        "question": "Find movies released between 2000 and 2010.",
        "expected": (
            r"\bm\.released\s*>=\s*2000\b"
            r".*\bAND\b.*"
            r"\bm\.released\s*<=\s*2010\b"
        ),
        "required": ["MATCH", "Movie", "WHERE"],
        "forbidden": [r"\bBETWEEN\b"]
    }
]

# Basic conservative screening; this is not a security parser.
WRITE_OR_ADMIN = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|"
    r"CALL|LOAD\s+CSV|FOREACH|GRANT|DENY|REVOKE)\b",
    re.IGNORECASE
)

ALLOWED_LABELS = {"Movie", "Person", "Genre"}
ALLOWED_RELATIONSHIPS = {"ACTED_IN", "DIRECTED", "IN_GENRE"}
ALLOWED_PROPERTIES = {"title", "released"}

def basic_balance_check(query):
    # Remove quoted strings before checking delimiter balance.
    text = re.sub(r"'(?:\\.|[^'\\])*'", "''", query)
    text = re.sub(r'"(?:\\.|[^"\\])*"', '""', text)

    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []

    for char in text:
        if char in "([{":
            stack.append(char)
        elif char in ")]}":
            if not stack or stack.pop() != pairs[char]:
                return False

    return not stack



def evaluate_query(query, test):
    query = query.strip()
    query = re.sub(
        r"```(?:cypher)?", "", query, flags=re.IGNORECASE
    ).replace("```", "").strip()

    statement = query.rstrip(";").strip()

    one_statement = ";" not in statement

    read_only_screen = (
        bool(statement)
        and not WRITE_OR_ADMIN.search(statement)
        and one_statement
    )

    starts_correctly = bool(
        re.match(r"^(MATCH|OPTIONAL\s+MATCH)\b",
                 statement, re.IGNORECASE)
    )

    expected_ok = bool(
        re.search(test["expected"], statement, re.IGNORECASE)
    )

    required_ok = all(
        re.search(rf"\b{re.escape(token)}\b",
                  statement, re.IGNORECASE)
        for token in test["required"]
    )

    forbidden_ok = all(
        not re.search(pattern, statement, re.IGNORECASE)
        for pattern in test["forbidden"]
    )

    balanced = basic_balance_check(statement)

    # Extract labels from node patterns, e.g. (m:Movie)
    labels_found = set(
        re.findall(
            r"\(\s*\w+\s*:\s*([A-Za-z_]\w*)",
            statement
        )
    )
    labels_ok = labels_found.issubset(ALLOWED_LABELS)

    # Extract relationship types, e.g. [:ACTED_IN]
    rels_found = set(
        re.findall(
            r"\[\s*:\s*([A-Za-z_]\w*)",
            statement
        )
    )
    relationships_ok = rels_found.issubset(
        ALLOWED_RELATIONSHIPS
    )

    passed = all([
        read_only_screen,
        starts_correctly,
        expected_ok,
        required_ok,
        forbidden_ok,
        balanced,
        labels_ok,
        relationships_ok
    ])

    issues = []
    if not read_only_screen:
        issues.append("Read-only/multiple-statement screen failed")
    if not starts_correctly:
        issues.append("Query must start with MATCH/OPTIONAL MATCH")
    if not expected_ok:
        issues.append("Expected pattern missing")
    if not required_ok:
        issues.append("Required clause or label missing")
    if not forbidden_ok:
        issues.append("Forbidden pattern present")
    if not balanced:
        issues.append("Unbalanced delimiters")
    if not labels_ok:
        issues.append(f"Undeclared node label: {labels_found - ALLOWED_LABELS}")
    if not relationships_ok:
        issues.append(
            f"Undeclared relationship: "
            f"{rels_found - ALLOWED_RELATIONSHIPS}"
        )

    return {
        "query": statement,
        "read_only_screen": read_only_screen,
        "expected_pattern": expected_ok,
        "balanced": balanced,
        "schema_labels_ok": labels_ok,
        "relationships_ok": relationships_ok,
        "result": "PASS" if passed else "REVIEW",
        "issues": "; ".join(issues) if issues
                  else "Basic checks passed"
    }


# Execute all 10 tests
results = []

for test in TESTS:
    print("\n" + "=" * 65)
    print(f"{test['id']}: {test['question']}")

    try:
        query = generate_cypher(test["question"])
        checks = evaluate_query(query, test)

        print("\nGenerated query:")
        print(checks["query"] or "[EMPTY OUTPUT]")
        print("\nExpected pattern:", checks["expected_pattern"])
        print("Read-only screen:", checks["read_only_screen"])
        print("Balanced delimiters:", checks["balanced"])
        print("Labels allowed:", checks["schema_labels_ok"])
        print("Relationships allowed:", checks["relationships_ok"])
        print("Result:", checks["result"])

        if checks["issues"] != "Basic checks passed":
            print("Issues:", checks["issues"])

        results.append({
            "ID": test["id"],
            "Question": test["question"],
            "Generated Cypher": checks["query"],
            "Expected pattern": checks["expected_pattern"],
            "Read-only screen": checks["read_only_screen"],
            "Balanced": checks["balanced"],
            "Labels OK": checks["schema_labels_ok"],
            "Relationships OK": checks["relationships_ok"],
            "Result": checks["result"],
            "Issues": checks["issues"]
        })

    except Exception as e:
        print("ERROR:", repr(e))
        results.append({
            "ID": test["id"],
            "Question": test["question"],
            "Generated Cypher": "",
            "Expected pattern": False,
            "Read-only screen": False,
            "Balanced": False,
            "Labels OK": False,
            "Relationships OK": False,
            "Result": "ERROR",
            "Issues": repr(e)
        })

df_results = pd.DataFrame(results)

print("\n" + "=" * 65)
print("QUERYCRAFT EVALUATION SUMMARY")
print("Passed:", (df_results["Result"] == "PASS").sum(), "/", len(TESTS))
print("Needs review:", (df_results["Result"] == "REVIEW").sum())
print("Errors:", (df_results["Result"] == "ERROR").sum())

display(df_results)


In [ ]:
import re

q08_pattern = r"\bcount\s*\(\s*(?:m|\*)\s*\)"

test_queries = [
    "MATCH (m:Movie) RETURN count(m)",
    "MATCH (m:Movie) RETURN COUNT(*)",
    "MATCH (m:Movie) RETURN count ( m )",
    "MATCH (m:Movie) RETURN count ( * )",
]

for query in test_queries:
    passed = bool(
        re.search(q08_pattern, query, re.IGNORECASE)
    )
    print("PASS" if passed else "FAIL", "-", query)

In [ ]:

import re
import gradio as gr

# QueryCraft: basic preview validation only.
# This does NOT prove a query is valid or safe to execute.

def querycraft_interface(question):
    if not question or not question.strip():
        return "", "Please enter a question."

    try:
        query = generate_cypher(question.strip())

        if not query or not query.strip():
            return "", "Generation failed: the model returned empty output."

        query = query.strip()

        # Basic screening for the preview stage only.
        blocked = re.search(
            r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|"
            r"CALL|LOAD\s+CSV|FOREACH|GRANT|DENY|REVOKE)\b",
            query,
            re.IGNORECASE
        )

        # Check for multiple statements.
        statement = query.rstrip(";").strip()
        multiple_statements = ";" in statement

        # Check delimiters, ignoring quoted strings.
        balance_text = re.sub(
            r"'(?:\\.|[^'\\])*'", "''", statement
        )
        balance_text = re.sub(
            r'"(?:\\.|[^"\\])*"', '""', balance_text
        )

        pairs = {")": "(", "]": "[", "}": "{"}
        stack = []
        balanced = True

        for char in balance_text:
            if char in "([{":
                stack.append(char)
            elif char in ")]}":
                if not stack or stack.pop() != pairs[char]:
                    balanced = False
                    break

        balanced = balanced and not stack

        if blocked or multiple_statements or not balanced:
            return query, (
                "REVIEW REQUIRED: basic screening detected a potential "
                "problem. Do not execute this query."
            )

        if not re.match(
            r"^(MATCH|OPTIONAL\s+MATCH)\b",
            statement,
            re.IGNORECASE
        ):
            return query, (
                "REVIEW REQUIRED: query does not begin with MATCH "
                "or OPTIONAL MATCH. Do not execute it."
            )

        return query, (
            "Generated. Basic checks passed; Cypher syntax, schema, "
            "semantics, and database safety have NOT been verified."
        )

    except Exception as e:
        return "", f"Generation error: {type(e).__name__}: {e}"


with gr.Blocks(title="QueryCraft - Natural Language to Cypher") as demo:
    gr.Markdown("# QueryCraft")
    gr.Markdown(
        "Translate natural-language questions into Neo4j Cypher. "
        "This initial version generates queries only; it does not "
        "connect to or execute queries against Neo4j."
    )

    question_input = gr.Textbox(
        label="Enter your question",
        placeholder="Example: What is the release year of The Matrix?",
        lines=2
    )

    generate_button = gr.Button(
        "Generate Cypher",
        variant="primary"
    )

    query_output = gr.Textbox(
        label="Generated Cypher",
        lines=6,
        interactive=False
    )

    status_output = gr.Textbox(
        label="Validation status",
        lines=3,
        interactive=False
    )

    generate_button.click(
        fn=querycraft_interface,
        inputs=question_input,
        outputs=[query_output, status_output]
    )

    question_input.submit(
        fn=querycraft_interface,
        inputs=question_input,
        outputs=[query_output, status_output]
    )

demo.launch(share=True)
